In [ ]:
# =================== SCSegamba-Lite (fixed dilation + new AMP API) ===================
# ✔ 不依賴 mamba-ssm / piq；純 PyTorch + albumentations + opencv
# ✔ 若 /content/crack_pairs 不存在，自動合成玩具資料（120 張）做 smoke test
# ✔ 真實資料結構：
#    /content/crack_pairs/{train,valid}/{images,masks}/*.png  (masks 為單通道 0/255)
# ================================================================================

# ---- 0) 安裝輕量依賴 ----
!pip -q install albumentations==1.4.7 opencv-python==4.10.0.84

# ---- 1) 匯入與基本設定 ----
import os, glob, random, math, time
from pathlib import Path
import numpy as np
from PIL import Image
import cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from tqdm import tqdm

DATA_ROOT = Path("/content/crack_pairs")
WORK_DIR  = Path("/content/work_scsegamba")
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEED=2025; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark=True

# ---- 2) 若沒有資料：自動合成小型裂縫資料集 ----
def synthesize_crack_image(H=512, W=512):
    img = (np.random.rand(H,W,3)*30+80).astype(np.uint8)
    img = cv2.GaussianBlur(img, (0,0), 3)
    mask = np.zeros((H,W), np.uint8)
    n_lines = np.random.randint(1, 4)
    for _ in range(n_lines):
        x1,y1 = np.random.randint(0,W), np.random.randint(0,H)
        x2,y2 = np.random.randint(0,W), np.random.randint(0,H)
        thickness = np.random.randint(1,3)
        cv2.line(mask, (x1,y1), (x2,y2), 255, thickness=thickness)
        for t in np.linspace(0,1,20):
            px=int(x1*(1-t)+x2*t + np.random.randn()*2)
            py=int(y1*(1-t)+y2*t + np.random.randn()*2)
            if 0<=px<W and 0<=py<H:
                mask[py,px]=255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    mask = cv2.dilate(mask, kernel, iterations=np.random.randint(0,2))
    mask = cv2.GaussianBlur(mask, (3,3), 0)
    mask = (mask>127).astype(np.uint8)*255
    crack = (mask==255)[...,None]
    noise = (np.random.randn(H,W,3)*10).astype(np.int16)
    out = img.astype(np.int16) - crack.astype(np.int16)*30 + noise
    out = np.clip(out, 0, 255).astype(np.uint8)
    return out, mask

def ensure_dataset(root=DATA_ROOT, n_train=100, n_valid=20, H=512, W=512):
    if root.exists() and any((root/"train"/"images").glob("*.png")):
        print(f"[info] 偵測到現有資料：{root}；跳過合成")
        return
    print("[info] 未偵測到資料，開始合成小型裂縫資料集...")
    for split, n in [("train", n_train), ("valid", n_valid)]:
        (root/split/"images").mkdir(parents=True, exist_ok=True)
        (root/split/"masks").mkdir(parents=True, exist_ok=True)
        for i in tqdm(range(n), desc=f"Synth {split}"):
            img, msk = synthesize_crack_image(H,W)
            cv2.imwrite(str(root/split/"images"/f"{i:05d}.png"), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(root/split/"masks"/f"{i:05d}.png"), msk)
    print("[info] 合成完成。")

ensure_dataset()

# ---- 3) Dataset ----
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def to_tensor_image(x_np):
    return torch.from_numpy(x_np.transpose(2,0,1)).float() / 255.0

def to_tensor_mask(m_np):
    return torch.from_numpy(m_np[None]).float() / 255.0  # [1,H,W]

class CrackPairs(Dataset):
    def __init__(self, root, split="train", crop=256):
        root = Path(root)/split
        img_dir = root/"images"
        self.X = sorted([str(p) for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y = [str((root/"masks"/Path(p).name)) for p in self.X]
        self.split=split; self.crop=crop
        self.tf_train = A.Compose([
            A.RandomCrop(crop, crop),
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.3),
            # Albumentations 會提醒可改 Affine；僅為 warning，功能可用
        ])
        self.tf_valid = A.Compose([])
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = np.array(Image.open(self.X[i]).convert("RGB"))
        y = np.array(Image.open(self.Y[i]).convert("L"))
        y = (y>127).astype(np.uint8)*255
        if self.split=="train":
            h,w=x.shape[:2]
            ph=max(0,self.crop-h); pw=max(0,self.crop-w)
            if ph or pw:
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y=cv2.copyMakeBorder(y,0,ph,0,pw,cv2.BORDER_REFLECT_101)
            aug=self.tf_train(image=x, mask=y); x,y=aug["image"], aug["mask"]
        return to_tensor_image(x), to_tensor_mask(y)

# ---- 4) 模組（修正：支援 dilation 的 same-padding）----
def conv_bn_act(cin, cout, k=3, s=1, dil=1, groups=1, act=nn.ReLU):
    # same padding：pad = ((k-1)//2)*dil
    pad = ((k - 1) // 2) * dil
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=k, stride=s, padding=pad, dilation=dil,
                  groups=groups, bias=False),
        nn.BatchNorm2d(cout),
        act(inplace=True)
    )

class GBC(nn.Module):
    def __init__(self, c, r=4, dilation=2):
        super().__init__()
        mid=max(c//r, 8)
        self.reduce = conv_bn_act(c, mid, k=1, dil=1)
        self.dw     = conv_bn_act(mid, mid, k=3, dil=dilation, groups=mid)  # ★ 正確 same-padding + dilation
        self.expand = conv_bn_act(mid, c,   k=1, dil=1)
        self.gate   = nn.Sequential(nn.Conv2d(c, c, 1), nn.Sigmoid())
    def forward(self, x):
        g = self.gate(x)
        h = self.expand(self.dw(self.reduce(x)))
        return x + h * g

class RowColGRU(nn.Module):
    def __init__(self, c, hidden=None, num_layers=1, bidir=True, dropout=0.0):
        super().__init__()
        hidden = hidden or c
        self.norm   = nn.BatchNorm2d(c)
        self.proj_in= nn.Conv2d(c, hidden, 1)
        self.row_rnn= nn.GRU(input_size=hidden, hidden_size=hidden//2 if bidir else hidden,
                             num_layers=num_layers, batch_first=True, bidirectional=bidir, dropout=dropout)
        self.col_rnn= nn.GRU(input_size=hidden, hidden_size=hidden//2 if bidir else hidden,
                             num_layers=num_layers, batch_first=True, bidirectional=bidir, dropout=dropout)
        out_c = hidden*2
        self.proj_out= nn.Conv2d(out_c, c, 1)
    def scan_row(self, x_h):
        B,C,H,W = x_h.shape
        seq = x_h.permute(0,2,3,1).reshape(B*H, W, C)
        out,_ = self.row_rnn(seq)
        out = out.reshape(B,H,W,-1).permute(0,3,1,2)
        return out
    def scan_col(self, x_h):
        B,C,H,W = x_h.shape
        seq = x_h.permute(0,3,2,1).reshape(B*W, H, C)
        out,_ = self.col_rnn(seq)
        out = out.reshape(B,W,H,-1).permute(0,3,2,1)
        return out
    def forward(self, x):
        x_n = self.norm(x)
        x_h = self.proj_in(x_n)
        r = self.scan_row(x_h)
        c = self.scan_col(x_h)
        y = torch.cat([r,c], dim=1)
        y = self.proj_out(y)
        return x + y

class SAVSS(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gbc = GBC(c)
        self.sass= RowColGRU(c)
    def forward(self, x):
        return self.sass(self.gbc(x))

# ---- 5) Network ----
class SCSegambaLite(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        c1,c2,c3,c4 = base, base*2, base*4, base*8
        self.e1 = conv_bn_act(3, c1, k=3)
        self.e2 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c1, c2, k=3))
        self.e3 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c2, c3, k=3))
        self.e4 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c3, c4, k=3))
        self.b1, self.b2, self.b3, self.b4 = SAVSS(c1), SAVSS(c2), SAVSS(c3), SAVSS(c4)
        self.up3 = nn.ConvTranspose2d(c4, c3, 2,2); self.d3 = conv_bn_act(c3*2, c3, k=3)
        self.up2 = nn.ConvTranspose2d(c3, c2, 2,2); self.d2 = conv_bn_act(c2*2, c2, k=3)
        self.up1 = nn.ConvTranspose2d(c2, c1, 2,2); self.d1 = conv_bn_act(c1*2, c1, k=3)
        self.ref = SAVSS(c1)
        self.head= nn.Conv2d(c1, 1, 1)
    def forward(self, x):
        f1 = self.b1(self.e1(x))
        f2 = self.b2(self.e2(f1))
        f3 = self.b3(self.e3(f2))
        f4 = self.b4(self.e4(f3))
        u3 = self.up3(f4); d3=self.d3(torch.cat([u3,f3],1))
        u2 = self.up2(d3); d2=self.d2(torch.cat([u2,f2],1))
        u1 = self.up1(d2); d1=self.d1(torch.cat([u1,f1],1))
        y  = self.head(self.ref(d1))
        return y  # logits

# ---- 6) Loss / Metrics ----
bce = nn.BCEWithLogitsLoss()

@torch.no_grad()
def binary_metrics(logits, y, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou = tp/(tp+fp+fn+eps); f1 = 2*tp/(2*tp+fp+fn+eps)
    return iou, f1

# ---- 7) Train / Val / 可視化（新 AMP API）----
def train(
    root=DATA_ROOT, work=WORK_DIR, base=32, crop=256, batch=8, epochs=20, lr=2e-4, amp=True
):
    device="cuda" if torch.cuda.is_available() else "cpu"
    tr=CrackPairs(root,"train",crop); va=CrackPairs(root,"valid",crop)
    tl=DataLoader(tr,batch,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    vl=DataLoader(va,1,shuffle=False,num_workers=2)
    model=SCSegambaLite(base).to(device)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model.__class__.__name__} | Params: {n_params/1e6:.2f}M")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(epochs,1))

    use_amp = amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    best_iou=-1; best=str(work/"scsegamba_best.pth")
    for ep in range(1,epochs+1):
        model.train(); pbar=tqdm(tl, desc=f"Epoch {ep}/{epochs}")
        for x,y in pbar:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits=model(x); loss=bce(logits,y)
            if use_amp:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        sch.step()
        # valid
        model.eval(); ious=[]; f1s=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits=model(x)
                iou,f1=binary_metrics(logits,y)
                ious.append(iou); f1s.append(f1)
        miou=float(np.mean(ious)); mf1=float(np.mean(f1s))
        print(f"[Val] mIoU={miou:.4f}  F1={mf1:.4f}")
        if miou>best_iou:
            best_iou=miou
            torch.save(model.state_dict(), best)
    print(f"Done. Best mIoU={best_iou:.4f} | saved to {best}")
    return model

@torch.no_grad()
def visualize_samples(model, root=DATA_ROOT, split="valid", n=6, out_dir=WORK_DIR/"viz"):
    device="cuda" if torch.cuda.is_available() else "cpu"
    ds=CrackPairs(root,split,crop=256)
    dl=DataLoader(ds,1,shuffle=False)
    model.eval(); out_dir=Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    k=0
    for x,y in dl:
        if k>=n: break
        x,y=x.to(device), y.to(device)
        logits=model(x); prob=torch.sigmoid(logits)
        grid = torch.cat([x, prob.repeat(1,3,1,1), y.repeat(1,3,1,1)], dim=0).clamp(0,1)
        grid_np = (grid.cpu().numpy().transpose(0,2,3,1)*255).astype(np.uint8)
        vis = np.vstack([grid_np[i] for i in range(grid_np.shape[0])])
        cv2.imwrite(str(out_dir/f"sample_{k:03d}.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        k+=1
    print(f"[viz] saved {k} samples to {out_dir}")

# ---- 8) 執行（T4 友善）----
model=train(base=32, crop=256, batch=8, epochs=20, lr=2e-4, amp=True)
visualize_samples(model, n=6)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.


[info] 偵測到現有資料：/content/crack_pairs；跳過合成
Model: SCSegambaLite | Params: 2.15M


Epoch 1/20: 100%|██████████| 12/12 [00:35<00:00,  2.94s/it, loss=0.6653]


[Val] mIoU=0.0060  F1=0.0119


Epoch 2/20: 100%|██████████| 12/12 [00:03<00:00,  3.54it/s, loss=0.5348]


...

[Val] mIoU=0.6790  F1=0.7682


Epoch 19/20: 100%|██████████| 12/12 [00:03<00:00,  3.57it/s, loss=0.0657]


[Val] mIoU=0.6853  F1=0.7725


Epoch 20/20: 100%|██████████| 12/12 [00:03<00:00,  3.59it/s, loss=0.0652]


[Val] mIoU=0.6877  F1=0.7740
Done. Best mIoU=0.6877 | saved to /content/work_scsegamba/scsegamba_best.pth
[viz] saved 6 samples to /content/work_scsegamba/viz


In [ ]:
# =================== SCSegamba-Lite + TUT Dataset (Colab) ===================
# ✔ 直接用作者釋出的 TUT 資料集：train_img/train_lab/val_img/val_lab/test_img/test_lab
# ✔ 不合成資料；純 PyTorch + Albumentations + OpenCV；新 AMP API
# ----------------------------------------------------------------------------

# ---- 0) 安裝依賴 ----
!pip -q install albumentations==1.4.7 opencv-python==4.10.0.84 gdown

# ---- 1) 下載並解壓 TUT 資料集（作者 Drive 連結）----
# 來源：Karl1109/TUT（README 的 Google Drive 連結）
# 若連不上，請在瀏覽器確認你能存取該 Drive 檔，或把檔案放到自己雲端後換 ID。
TUT_ID = "1A-LOJzY-KLsrNCBF8MerqflGCKoiBKF-"  # TUT.zip
!gdown -q --id "{TUT_ID}" -O /content/TUT.zip
!unzip -q -o /content/TUT.zip -d /content/
!ls -al /content/TUT | head -n 20

# ---- 2) 匯入與基本設定 ----
import os, glob, random, math, time
from pathlib import Path
import numpy as np
from PIL import Image
import cv2

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from tqdm import tqdm

DATA_ROOT = Path("/content/TUT")  # 解壓後的資料夾（內含 train_img/train_lab/...）
WORK_DIR  = Path("/content/work_scsegamba_tut"); WORK_DIR.mkdir(parents=True, exist_ok=True)
SEED=2025; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark=True

# ---- 3) TUT Dataset Loader --------------------------------------------------
# 結構（出自作者 TUT 倉庫 README）：
#   TUT/
#     train_img/*.jpg; train_lab/*.png
#     val_img/*.jpg;   val_lab/*.png
#     test_img/*.jpg;  test_lab/*.png
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def _imread_rgb(p):  return np.array(Image.open(p).convert("RGB"))
def _imread_mask(p): return (np.array(Image.open(p).convert("L")) > 127).astype(np.uint8)*255

class TUTCrackDataset(Dataset):
    def __init__(self, root, split="train", crop=256):
        root = Path(root)
        assert split in {"train","val","test"}
        self.img_dir = root/f"{split}_img"
        self.lab_dir = root/f"{split}_lab"
        # TUT 影像副檔名多為 .jpg，標註為 .png（同檔名不同副檔名）
        self.X = sorted([p for p in self.img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y = [self.lab_dir/f"{Path(p).stem}.png" for p in self.X]
        self.split=split; self.crop=crop
        # Train：隨機裁切 + 幾何/顏色增強；Val/Test：僅轉張量（不強制裁切）
        self.tf_train = A.Compose([
            A.RandomCrop(crop, crop),
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.30),
            A.ColorJitter(p=0.20, brightness=0.10, contrast=0.10, saturation=0.10, hue=0.02),
        ])
        self.tf_eval = A.Compose([])

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        xi, yi = str(self.X[i]), str(self.Y[i])
        x = _imread_rgb(xi)
        y = _imread_mask(yi)
        if self.split == "train":
            h,w = x.shape[:2]
            ph=max(0,self.crop-h); pw=max(0,self.crop-w)
            if ph or pw:
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y=cv2.copyMakeBorder(y,0,ph,0,pw,cv2.BORDER_REFLECT_101)
            aug=self.tf_train(image=x, mask=y); x,y=aug["image"], aug["mask"]
        # 轉 tensor（CHW, float32）
        x = torch.from_numpy(x.transpose(2,0,1)).float()/255.0
        y = torch.from_numpy(y[None]).float()/255.0
        return x, y

# ---- 4) 模組（支援 dilation 的 same-padding）-----------------------------------
def conv_bn_act(cin, cout, k=3, s=1, dil=1, groups=1, act=nn.ReLU):
    pad = ((k - 1) // 2) * dil
    return nn.Sequential(
        nn.Conv2d(cin, cout, kernel_size=k, stride=s, padding=pad, dilation=dil,
                  groups=groups, bias=False),
        nn.BatchNorm2d(cout),
        act(inplace=True)
    )

class GBC(nn.Module):
    def __init__(self, c, r=4, dilation=2):
        super().__init__()
        mid=max(c//r, 8)
        self.reduce = conv_bn_act(c, mid, k=1, dil=1)
        self.dw     = conv_bn_act(mid, mid, k=3, dil=dilation, groups=mid)  # depthwise + dilation
        self.expand = conv_bn_act(mid, c,   k=1, dil=1)
        self.gate   = nn.Sequential(nn.Conv2d(c, c, 1), nn.Sigmoid())
    def forward(self, x):
        g = self.gate(x)
        h = self.expand(self.dw(self.reduce(x)))
        return x + h * g

class RowColGRU(nn.Module):
    def __init__(self, c, hidden=None, num_layers=1, bidir=True, dropout=0.0):
        super().__init__()
        hidden = hidden or c
        self.norm   = nn.BatchNorm2d(c)
        self.proj_in= nn.Conv2d(c, hidden, 1)
        self.row_rnn= nn.GRU(input_size=hidden, hidden_size=hidden//2 if bidir else hidden,
                             num_layers=num_layers, batch_first=True, bidirectional=bidir, dropout=dropout)
        self.col_rnn= nn.GRU(input_size=hidden, hidden_size=hidden//2 if bidir else hidden,
                             num_layers=num_layers, batch_first=True, bidirectional=bidir, dropout=dropout)
        out_c = hidden*2
        self.proj_out= nn.Conv2d(out_c, c, 1)
    def _scan_row(self, x_h):
        B,C,H,W = x_h.shape
        seq = x_h.permute(0,2,3,1).reshape(B*H, W, C)
        out,_ = self.row_rnn(seq)
        out = out.reshape(B,H,W,-1).permute(0,3,1,2)
        return out
    def _scan_col(self, x_h):
        B,C,H,W = x_h.shape
        seq = x_h.permute(0,3,2,1).reshape(B*W, H, C)
        out,_ = self.col_rnn(seq)
        out = out.reshape(B,W,H,-1).permute(0,3,2,1)
        return out
    def forward(self, x):
        x_n = self.norm(x)
        x_h = self.proj_in(x_n)
        r = self._scan_row(x_h)
        c = self._scan_col(x_h)
        y = torch.cat([r,c], dim=1)
        y = self.proj_out(y)
        return x + y

class SAVSS(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gbc = GBC(c)
        self.sass= RowColGRU(c)
    def forward(self, x):
        return self.sass(self.gbc(x))

# ---- 5) 網路（加入安全對齊，避免奇數尺寸 concat 失配）-------------------------
def cat_align(up, skip):
    # 以雙線性插值把 skip 對齊到 up 的尺寸再 concat
    if up.shape[-2:] != skip.shape[-2:]:
        skip = F.interpolate(skip, size=up.shape[-2:], mode='bilinear', align_corners=False)
    return torch.cat([up, skip], 1)

class SCSegambaLite(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        c1,c2,c3,c4 = base, base*2, base*4, base*8
        self.e1 = conv_bn_act(3, c1, k=3)
        self.e2 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c1, c2, k=3))
        self.e3 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c2, c3, k=3))
        self.e4 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c3, c4, k=3))
        self.b1, self.b2, self.b3, self.b4 = SAVSS(c1), SAVSS(c2), SAVSS(c3), SAVSS(c4)
        self.up3 = nn.ConvTranspose2d(c4, c3, 2,2); self.d3 = conv_bn_act(c3*2, c3, k=3)
        self.up2 = nn.ConvTranspose2d(c3, c2, 2,2); self.d2 = conv_bn_act(c2*2, c2, k=3)
        self.up1 = nn.ConvTranspose2d(c2, c1, 2,2); self.d1 = conv_bn_act(c1*2, c1, k=3)
        self.ref = SAVSS(c1)
        self.head= nn.Conv2d(c1, 1, 1)
    def forward(self, x):
        f1 = self.b1(self.e1(x))
        f2 = self.b2(self.e2(f1))
        f3 = self.b3(self.e3(f2))
        f4 = self.b4(self.e4(f3))
        u3 = self.up3(f4); d3=self.d3(cat_align(u3, f3))
        u2 = self.up2(d3); d2=self.d2(cat_align(u2, f2))
        u1 = self.up1(d2); d1=self.d1(cat_align(u1, f1))
        y  = self.head(self.ref(d1))
        return y  # logits

# ---- 6) Loss / Metrics ------------------------------------------------------
bce = nn.BCEWithLogitsLoss()

@torch.no_grad()
def binary_metrics(logits, y, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou = tp/(tp+fp+fn+eps); f1 = 2*tp/(2*tp+fp+fn+eps)
    return iou, f1

# ---- 7) Train / Val / 可視化（新 AMP API）--------------------------------------
def train_tut(
    root=DATA_ROOT, work=WORK_DIR, base=32, crop=256, batch=8, epochs=20, lr=2e-4, amp=True
):
    device="cuda" if torch.cuda.is_available() else "cpu"
    tr=TUTCrackDataset(root,"train",crop); va=TUTCrackDataset(root,"val",crop)
    tl=DataLoader(tr,batch,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    vl=DataLoader(va,1,shuffle=False,num_workers=2)
    model=SCSegambaLite(base).to(device)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model.__class__.__name__} | Params: {n_params/1e6:.2f}M")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(epochs,1))

    use_amp = amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    best_iou=-1; best=str(work/"scsegamba_tut_best.pth")
    for ep in range(1,epochs+1):
        model.train(); pbar=tqdm(tl, desc=f"Epoch {ep}/{epochs}")
        for x,y in pbar:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits=model(x); loss=bce(logits,y)
            if use_amp:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        sch.step()
        # valid
        model.eval(); ious=[]; f1s=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits=model(x)
                iou,f1=binary_metrics(logits,y)
                ious.append(iou); f1s.append(f1)
        miou=float(np.mean(ious)); mf1=float(np.mean(f1s))
        print(f"[Val@TUT] mIoU={miou:.4f}  F1={mf1:.4f}")
        if miou>best_iou:
            best_iou=miou
            torch.save(model.state_dict(), best)
    print(f"Done. Best mIoU={best_iou:.4f} | saved to {best}")
    return model

@torch.no_grad()
def visualize_samples(model, root=DATA_ROOT, split="val", n=6, out_dir=WORK_DIR/"viz_tut"):
    device="cuda" if torch.cuda.is_available() else "cpu"
    ds=TUTCrackDataset(root,split,crop=256)
    dl=DataLoader(ds,1,shuffle=False)
    model.eval(); out_dir=Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    k=0
    for x,y in dl:
        if k>=n: break
        x,y=x.to(device), y.to(device)
        logits=model(x); prob=torch.sigmoid(logits)
        grid = torch.cat([x, prob.repeat(1,3,1,1), y.repeat(1,3,1,1)], dim=0).clamp(0,1)
        grid_np = (grid.cpu().numpy().transpose(0,2,3,1)*255).astype(np.uint8)
        vis = np.vstack([grid_np[i] for i in range(grid_np.shape[0])])
        cv2.imwrite(str(out_dir/f"sample_{k:03d}.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        k+=1
    print(f"[viz] saved {k} samples to {out_dir}")

# ---- 8) 執行 ----
assert DATA_ROOT.exists(), f"找不到 {DATA_ROOT}（請確認 TUT.zip 已正確解壓）"
model=train_tut(base=32, crop=256, batch=8, epochs=20, lr=2e-4, amp=True)
visualize_samples(model, split="val", n=6)


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
total 2116
drwxrwxrwx 8 root root    4096 Oct 28  2024 .
drwxr-xr-x 1 root root    4096 Nov 17 11:53 ..
-rw-rw-rw- 1 root root 2046803 Oct 17  2024 Preview.png
-rw-rw-rw- 1 root root    1097 Oct 28  2024 README.md
drwxrwxrwx 2 root root   12288 Aug 22  2024 test_img
drwxrwxrwx 2 root root   12288 Aug 22  2024 test_lab
drwxrwxrwx 2 root root   36864 Aug 22  2024 train_img
drwxrwxrwx 2 root root   36864 Aug 22  2024 train_lab
drwxrwxrwx 2 root root    4096 Aug 22  2024 val_img
drwxrwxrwx 2 root root    4096 Aug 22  2024 val_lab
Model: SCSegambaLite | Params: 2.15M


Epoch 1/20: 100%|██████████| 123/123 [00:33<00:00,  3.73it/s, loss=0.1589]


[Val@TUT] mIoU=0.2708  F1=0.3962


Epoch 2/20: 100%|██████████| 123/123 [00:31<00:00,  3.86it/s, loss=0.0808]


[Val@TUT] mIoU=0.4450  F1=0.5875


...

[Val@TUT] mIoU=0.5979  F1=0.7390


Epoch 19/20: 100%|██████████| 123/123 [00:31<00:00,  3.85it/s, loss=0.0429]


[Val@TUT] mIoU=0.6013  F1=0.7423


Epoch 20/20: 100%|██████████| 123/123 [00:31<00:00,  3.86it/s, loss=0.0505]


[Val@TUT] mIoU=0.5957  F1=0.7374
Done. Best mIoU=0.6013 | saved to /content/work_scsegamba_tut/scsegamba_tut_best.pth
[viz] saved 6 samples to /content/work_scsegamba_tut/viz_tut


In [ ]:
# ===================== SCSegamba (from-scratch, paper-faithful) =====================
# ✅ 與論文一致的模組：SAVSS = GBC + SASS(SS2D/對角蛇形)，MFS = per-level MLP + DySample + Concat + GBC + MLP
# ✅ Loss：L = α·Dice + β·BCE，α:β = 1:5（論文設定）
# ✅ 資料：TUT（作者公開，目錄 train_img/train_lab/val_img/val_lab/test_img/test_lab）
# ✅ A100 友善：SS2D 視窗化（預設 16x16），節省記憶體；仍保留二維蛇形掃描的連續性
# ===================================================================================

# ---- 0) 安裝依賴（建議 Colab A100 環境）----
!pip -q install --extra-index-url https://download.pytorch.org/whl/cu121 \
    torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1
!pip -q install mamba-ssm[causal-conv1d]==2.2.4 albumentations==1.4.7 opencv-python==4.10.0.84 gdown

import os, glob, math, random, time
from pathlib import Path
import numpy as np
from PIL import Image
import cv2

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from tqdm import tqdm
from mamba_ssm import Mamba

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# ---- 1) 下載/放置 TUT 資料（作者提供）----
DATA_ROOT = Path("/content/TUT")
if not (DATA_ROOT/"train_img").exists():
    TUT_ID = "1A-LOJzY-KLsrNCBF8MerqflGCKoiBKF-"   # 官方 TUT 的 Google Drive 壓縮檔
    !gdown -q --id {TUT_ID} -O /content/TUT.zip
    !unzip -q -o /content/TUT.zip -d /content/
    # 如果解壓不在 /content/TUT，嘗試搬運
    import glob, shutil
    if not (DATA_ROOT/"train_img").exists():
        cand = None
        for p in glob.glob("/content/**/train_img", recursive=True):
            cand = Path(p).parent; break
        if cand and cand != DATA_ROOT:
            shutil.move(str(cand), str(DATA_ROOT))

assert (DATA_ROOT/"train_img").exists(), "TUT 資料夾不存在，請自行放置到 /content/TUT/*_img/*_lab"

# ---- 2) Dataset（TUT 結構）----
IMG_EXT = {".png",".jpg",".jpeg",".bmp",".webp"}
def _imread_rgb(p):  return np.array(Image.open(p).convert("RGB"))
def _imread_mask(p): return (np.array(Image.open(p).convert("L")) > 127).astype(np.uint8)*255

class TUTCrack(Dataset):
    def __init__(self, root, split="train", crop=256):
        self.root = Path(root); self.split = split; self.crop=crop
        assert split in {"train","val","test"}
        self.img_dir = self.root/f"{split}_img"
        self.lab_dir = self.root/f"{split}_lab"
        self.X = sorted([p for p in self.img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y = [self.lab_dir/f"{Path(p).stem}.png" for p in self.X]
        self.tf_train = A.Compose([
            A.RandomCrop(crop, crop),
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.2),
            A.ShiftScaleRotate(0.05, 0.1, 10, p=0.3),
            A.ColorJitter(p=0.2, brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
        ])
        self.tf_eval = A.Compose([])
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = _imread_rgb(str(self.X[i])); y = _imread_mask(str(self.Y[i]))
        if self.split=="train":
            h,w=x.shape[:2]; ph=max(0,self.crop-h); pw=max(0,self.crop-w)
            if ph or pw:
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y=cv2.copyMakeBorder(y,0,ph,0,pw,cv2.BORDER_REFLECT_101)
            aug=self.tf_train(image=x, mask=y); x,y=aug["image"], aug["mask"]
        # to tensor
        x = torch.from_numpy(x.transpose(2,0,1)).float()/255.0
        y = torch.from_numpy(y[None]).float()/255.0
        return x,y

# ---- 3) 位置編碼（簡化：可學參數的 2D sinusoidal）----
class SinePosEnc2D(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.proj = nn.Conv2d(2, c, 1)
    def forward(self, x):
        B,C,H,W = x.shape
        yy, xx = torch.linspace(-1,1,H,device=x.device), torch.linspace(-1,1,W,device=x.device)
        grid_y, grid_x = torch.meshgrid(yy, xx, indexing='ij')
        feat = torch.stack([torch.sin(math.pi*grid_x), torch.sin(math.pi*grid_y)], 0)[None]  # [1,2,H,W]
        return x + self.proj(feat.expand(B,-1,-1,-1))

# ---- 4) GBC（論文 3.2：瓶頸卷積 + 門控；此處使用 GroupNorm + 深度可分離 + 殘差）----
def conv_act(cin, cout, k=3, s=1, p=None, groups=1):
    if p is None: p = (k-1)//2
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, p, groups=groups, bias=False),
                         nn.GroupNorm(num_groups=min(8, cout), num_channels=cout),
                         nn.SiLU(inplace=True))

class GBC(nn.Module):
    def __init__(self, c, r=4, dilation=2):
        super().__init__()
        mid = max(c//r, 8)
        self.g1 = conv_act(c, c, 1, 1, 0)
        self.g2 = conv_act(c, c, 3, 1, dilation, groups=c)  # depthwise w/ dilation
        self.bott1 = conv_act(c, mid, 1, 1, 0)
        self.dw     = conv_act(mid, mid, 3, 1, dilation, groups=mid)
        self.bott2 = conv_act(mid, c, 1, 1, 0)
        self.gate  = nn.Sequential(nn.Conv2d(c, c, 1), nn.Sigmoid())
    def forward(self, x):
        # 生成門控特徵 m(x) = g1(x) ⊙ g2(x)（論文式(6)）；再經瓶頸卷積細化，最後殘差
        m = self.g1(x) * self.g2(x)
        y = self.bott2(self.dw(self.bott1(m)))
        y = y * self.gate(x)
        return x + y

# ---- 5) SS2D（SASS：對角「蛇形」掃描 + Mamba；視窗化以利計算）----
def build_diag_snake_index(h, w, anti=False):
    """
    產生 HxW 的「對角蛇形」掃描順序索引（長度 L=H*W），並回傳順序 idx 與其反向 idx_inv。
    anti=False: 主對角線族（i-j 常數）；anti=True: 反對角線族（i+j 常數）。
    """
    order = []
    if not anti:
        # 主對角線 i-j = const，從 -(h-1) 到 (w-1)
        for d in range(-(h-1), w):
            cells = []
            for i in range(h):
                j = i - d
                if 0 <= j < w:
                    cells.append((i,j))
            # 蛇形：每個對角線交替反轉
            if (d - (-(h-1))) % 2 == 1:
                cells.reverse()
            order.extend(cells)
    else:
        # 反對角線 i+j = const，從 0 到 h+w-2
        for s in range(h+w-1):
            cells = []
            for i in range(h):
                j = s - i
                if 0 <= j < w:
                    cells.append((i,j))
            if s % 2 == 1:
                cells.reverse()
            order.extend(cells)
    idx = [i*w + j for (i,j) in order]
    idx = np.array(idx, dtype=np.int64)
    idx_inv = np.zeros_like(idx)
    idx_inv[idx] = np.arange(h*w, dtype=np.int64)
    return idx, idx_inv

class SS2D_DiagSnake_Window(nn.Module):
    """
    二維選擇性掃描（SS2D）：以視窗（ws×ws）做「主對角蛇形」與「反對角蛇形」兩個掃描，
    各過一層 Mamba，最後 concat → 1×1 fuse。視窗化保留拓撲連續性又節省記憶體。
    """
    def __init__(self, c, ws=16, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.ws = ws
        self.mamba_main = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)
        self.mamba_anti = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)
        self.proj = nn.Conv2d(2*c, c, 1)
        # 預先建立單一視窗的索引
        idx_m, inv_m = build_diag_snake_index(ws, ws, anti=False)
        idx_a, inv_a = build_diag_snake_index(ws, ws, anti=True)
        self.register_buffer("idx_main", torch.from_numpy(idx_m))
        self.register_buffer("inv_main", torch.from_numpy(inv_m))
        self.register_buffer("idx_anti", torch.from_numpy(idx_a))
        self.register_buffer("inv_anti", torch.from_numpy(inv_a))

    def scan_once(self, x, idx, inv_idx, mamba):
        B,C,H,W = x.shape; ws=self.ws
        assert H%ws==0 and W%ws==0, "H,W 必須為視窗大小的整數倍（可先 F.pad 再裁切）"
        # 拆成視窗： [B, C, H/ws, W/ws, ws, ws] → [B*nwin, ws*ws, C]
        xw = x.view(B, C, H//ws, ws, W//ws, ws).permute(0,2,4,3,5,1).contiguous()
        nB = B*(H//ws)*(W//ws)
        xw = xw.view(nB, ws*ws, C)                       # row-major
        # 依對角蛇形順序重排
        xw = xw[:, idx]                                   # [nB, L, C]
        y,_ = mamba(xw)                                   # 同長度序列
        # 還原回視窗 row-major
        y = y[:, inv_idx]                                 # 反置換
        y = y.view(B, H//ws, W//ws, ws, ws, C).permute(0,5,1,3,2,4).contiguous()
        y = y.view(B, C, H, W)
        return y

    def forward(self, x):
        y1 = self.scan_once(x, self.idx_main, self.inv_main, self.mamba_main)
        y2 = self.scan_once(x, self.idx_anti, self.inv_anti, self.mamba_anti)
        return self.proj(torch.cat([y1,y2], dim=1))

# ---- 6) SAVSS：GBC → 位置編碼 → SS2D（對角蛇形） + 殘差 ----
class SAVSS(nn.Module):
    def __init__(self, c, ws=16):
        super().__init__()
        self.gbc = GBC(c)
        self.pe  = SinePosEnc2D(c)
        self.ss2d= SS2D_DiagSnake_Window(c, ws=ws)
        self.norm= nn.GroupNorm(num_groups=min(8, c), num_channels=c)
    def forward(self, x):
        r = x
        x = self.gbc(x)
        x = self.pe(x)
        x = self.ss2d(self.norm(x))
        return r + x

# ---- 7) MFS：每層 MLP → DySample 到原尺寸 → Concat → GBC → MLP → 1-channel ----
class PerPixelMLP(nn.Module):
    def __init__(self, c, hidden=None):
        super().__init__()
        hidden = hidden or c*2
        self.net = nn.Sequential(nn.Conv2d(c, hidden, 1), nn.SiLU(inplace=True), nn.Conv2d(hidden, c, 1))
    def forward(self, x): return self.net(x)

class DyUp2x(nn.Module):
    """
    輕量 2× 動態上採樣：產生 2x2 權重，對每個像素做局部動態重建；串聯重複達到 4×/8×。
    """
    def __init__(self, c):
        super().__init__()
        self.weight = nn.Conv2d(c, 4, 1)  # 產生每個像素對應 2x2 權重
    def forward(self, x):
        B,C,H,W = x.shape
        w = torch.softmax(self.weight(x), dim=1)  # [B,4,H,W]
        # 當作鄰近像素的動態重建（這裡採簡化：均勻分配到 2x2 子像素）
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        return x * F.interpolate(w.sum(1, keepdim=True), scale_factor=2, mode="bilinear", align_corners=False)

class DySampleUp(nn.Module):
    def __init__(self, c, scale):
        super().__init__()
        self.blocks = nn.ModuleList([DyUp2x(c) for _ in range(int(math.log2(scale)))]) if scale>1 else nn.ModuleList()
    def forward(self, x):
        for b in self.blocks: x = b(x)
        return x

class MFSHead(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.mlp1, self.mlp2, self.mlp3, self.mlp4 = PerPixelMLP(c), PerPixelMLP(c), PerPixelMLP(c), PerPixelMLP(c)
        self.up2 = DySampleUp(c, 2); self.up4 = DySampleUp(c, 4); self.up8 = DySampleUp(c, 8)
        self.fuse_gbc = GBC(c*4, r=2, dilation=1)  # 融合前再經 GBC（對應公式 (14)）
        self.out_mlp  = nn.Sequential(nn.Conv2d(c*4, c, 1), nn.SiLU(inplace=True), nn.Conv2d(c, 1, 1))
    def forward(self, f1,f2,f3,f4):
        F1 = self.mlp1(f1)              # H×W
        F2 = self.up2(self.mlp2(f2))    # ×2 → H×W
        F3 = self.up4(self.mlp3(f3))    # ×4 → H×W
        F4 = self.up8(self.mlp4(f4))    # ×8 → H×W
        cat = torch.cat([F1,F2,F3,F4], dim=1)
        cat = self.fuse_gbc(cat)
        out = self.out_mlp(cat)         # 1×H×W
        return out

# ---- 8) 主網路：Patch Embedding → SAVSS×4（每層下採樣）→ MFS ----
def down(c_in, c_out): return nn.Sequential(nn.Conv2d(c_in, c_out, 3, 2, 1, bias=False),
                                            nn.GroupNorm(num_groups=min(8,c_out), num_channels=c_out),
                                            nn.SiLU(inplace=True))
class SCSegamba(nn.Module):
    """
    論文圖 2 結構：輸入 → Patch Embedding/PE → SAVSS×4（每層下採樣）→ MFS Head（Concat→GBC→MLP）→ 輸出
    通道配置：F1/F2/F3/F4 ≈ 16/32/64/128（見論文圖示），可依據顯存調整 base。
    """
    def __init__(self, base=16, ws=16):
        super().__init__()
        c1,c2,c3,c4 = base, base*2, base*4, base*8
        self.embed = nn.Sequential(nn.Conv2d(3, c1, 3, 1, 1, bias=False),
                                   nn.GroupNorm(num_groups=min(8,c1), num_channels=c1),
                                   nn.SiLU(inplace=True))
        self.s1 = SAVSS(c1, ws)
        self.d1 = down(c1, c2); self.s2 = SAVSS(c2, ws)
        self.d2 = down(c2, c3); self.s3 = SAVSS(c3, ws)
        self.d3 = down(c3, c4); self.s4 = SAVSS(c4, ws)
        self.mfs = MFSHead(c1)  # MFS 最終輸出於原解析度
    def forward(self, x):
        f1 = self.s1(self.embed(x))
        f2 = self.s2(self.d1(f1))
        f3 = self.s3(self.d2(f2))
        f4 = self.s4(self.d3(f3))
        y  = self.mfs(f1,f2,f3,f4)
        return y

# ---- 9) Loss（論文：L = α·Dice + β·BCE，α:β=1:5）----
def dice_loss(logits, y, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2*(p*y).sum(dim=(2,3)) + eps
    den = (p.pow(2)+y.pow(2)).sum(dim=(2,3)) + eps
    return 1.0 - (num/den).mean()

class DiceBCELoss(nn.Module):
    def __init__(self, alpha=1.0, beta=5.0):
        super().__init__(); self.alpha=alpha; self.beta=beta
    def forward(self, logits, y):
        bce = F.binary_cross_entropy_with_logits(logits, y)
        dl  = dice_loss(logits, y)
        return self.alpha*dl + self.beta*bce

@torch.no_grad()
def binary_metrics(logits, y, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou = tp/(tp+fp+fn+eps); f1 = 2*tp/(2*tp+fp+fn+eps)
    return iou, f1

# ---- 10) 訓練與驗證 ----
def train_tut(
    root=DATA_ROOT, base=16, ws=16, crop=256, batch=8, epochs=20, lr=5e-4, wd=1e-2, amp=True
):
    device="cuda" if torch.cuda.is_available() else "cpu"
    tr=TUTCrack(root,"train",crop); va=TUTCrack(root,"val",crop)
    tl=DataLoader(tr,batch,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    vl=DataLoader(va,1,shuffle=False,num_workers=2)
    model=SCSegamba(base=base, ws=ws).to(device)
    n_params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {model.__class__.__name__} | Params: {n_params/1e6:.2f}M")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched=torch.optim.lr_scheduler.PolynomialLR(opt, total_iters=max(epochs,1))  # PolyLR（論文）
    use_amp = amp and torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    criterion = DiceBCELoss(alpha=1.0, beta=5.0).to(device)

    best_iou=-1; best="/content/scsegamba_tut_best.pth"
    for ep in range(1,epochs+1):
        model.train(); pbar=tqdm(tl, desc=f"Epoch {ep}/{epochs}")
        for x,y in pbar:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits=model(x); loss=criterion(logits,y)
            if use_amp:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
        sched.step()
        # valid
        model.eval(); ious=[]; f1s=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits=model(x)
                iou,f1=binary_metrics(logits,y)
                ious.append(iou); f1s.append(f1)
        miou=float(np.mean(ious)); mf1=float(np.mean(f1s))
        print(f"[Val@TUT] mIoU={miou:.4f}  F1={mf1:.4f}")
        if miou>best_iou:
            best_iou=miou; torch.save(model.state_dict(), best)
    print(f"Done. Best mIoU={best_iou:.4f} | saved: {best}")
    return model

@torch.no_grad()
def visualize_samples(model, root=DATA_ROOT, split="val", n=6, out_dir="/content/viz_tut"):
    device="cuda" if torch.cuda.is_available() else "cpu"
    ds=TUTCrack(root,split,crop=256); dl=DataLoader(ds,1,shuffle=False)
    model.eval(); Path(out_dir).mkdir(parents=True, exist_ok=True)
    k=0
    for x,y in dl:
        if k>=n: break
        x,y=x.to(device), y.to(device)
        pr=torch.sigmoid(model(x))
        grid=torch.cat([x, pr.repeat(1,3,1,1), y.repeat(1,3,1,1)], 0).clamp(0,1)
        g=(grid.cpu().numpy().transpose(0,2,3,1)*255).astype(np.uint8)
        vis=np.vstack([g[i] for i in range(g.shape[0])])
        cv2.imwrite(f"{out_dir}/sample_{k:03d}.png", cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))
        k+=1
    print(f"[viz] saved {k} samples to {out_dir}")

# ---- 11) 直接開訓（A100 建議 batch=8/epochs=20，ws=16；若顯存不足可把 batch 調小、ws 調大 16→32）----
model = train_tut(base=16, ws=16, crop=256, batch=8, epochs=20, lr=5e-4, wd=1e-2, amp=True)
visualize_samples(model, n=6)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 97.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 107.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 45.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 125.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━

ModuleNotFoundError: No module named 'mamba_ssm'

In [ ]:
!pip install mamba_ssm

  Using cached mamba_ssm-2.2.6.post3-cp312-cp312-linux_x86_64.whl


In [ ]:
# =================== SCSegamba (CVPR'25) — single-file, paper-aligned ===================
# 架構對齊論文 Fig.2 與式(4)–(16)：GBC、SASS/SS2D、PAF、MFS(DySample)、BCE+Dice(1:5)
# Colab: 直接執行。先把資料準備成 /content/TUT/{train,val}/{images,masks}/*.png
# ======================================================================================

# ---- 0) 安裝依賴（Mamba SSM + 影像工具）----
# !pip -q install "mamba-ssm>=1.2.0" "causal-conv1d>=1.4.0" \
#                  albumentations==1.4.7 opencv-python==4.10.0.84 einops==0.8.0

# ---- 1) 匯入與設定 ----
import os, glob, math, random, time
from pathlib import Path
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from einops import rearrange

from mamba_ssm import Mamba   # 視覺 SSM 主角（對應文中 SAVSS 的 SSM 內核）

SEED=2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

# ---- 2) Dataset（TUT 風格；mask 必須是單通道 0/255）----
IMG_EXT = {".png",".jpg",".jpeg",".bmp",".webp"}

class CrackSegSet(Dataset):
    def __init__(self, root, split="train", crop=512):
        root = Path(root)/split
        img_dir, msk_dir = root/"images", root/"masks"
        self.X = sorted([str(p) for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y = [str(msk_dir/Path(p).name) for p in self.X]
        self.split, self.crop = split, crop
        self.tf_train = A.Compose([
            A.RandomCrop(crop, crop),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=10, p=0.30),
            A.RandomBrightnessContrast(0.1,0.1,p=0.3),
        ])
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = np.array(Image.open(self.X[i]).convert("RGB"))
        y = np.array(Image.open(self.Y[i]).convert("L"))
        y = (y>127).astype(np.uint8)*255
        if self.split=="train":
            h,w=x.shape[:2]; ph=max(0,self.crop-h); pw=max(0,self.crop-w)
            if ph or pw:
                x = cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y = cv2.copyMakeBorder(y,0,ph,0,pw,cv2.BORDER_REFLECT_101)
            aug = self.tf_train(image=x, mask=y); x,y = aug["image"], aug["mask"]
        x = torch.from_numpy(x.transpose(2,0,1)).float()/255.0
        y = torch.from_numpy(y[None]).float()/255.0
        return x, y

# ---- 3) 積木：Conv+Norm+Act、位置編碼（Fig.2 Encoder 前的 PE）----
def conv_bn_act(cin, cout, k=3, s=1, d=1, groups=1, act=nn.SiLU):
    pad = ((k-1)//2)*d
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, s, pad, dilation=d, groups=groups, bias=False),
        nn.BatchNorm2d(cout),
        act(inplace=True)
    )

class PosEnc2D(nn.Module):
    # 簡單 2D sine-cos 編碼，加到特徵上（對應 Fig.2 Encoder→PE）
    def __init__(self, C, temperature=10000.0):
        super().__init__()
        self.C = C
        self.temperature = temperature
    def forward(self, x):
        B,C,H,W = x.shape
        y, xg = torch.meshgrid(
            torch.arange(H, device=x.device),
            torch.arange(W, device=x.device),
            indexing='ij'
        )
        omega = torch.arange(C//4, device=x.device).float()
        omega = 1.0 / (self.temperature ** (2*omega/(C//2)))
        y = y[...,None]*omega; xg = xg[...,None]*omega
        pe = torch.cat([torch.sin(xg), torch.cos(xg), torch.sin(y), torch.cos(y)], dim=-1)
        pe = pe.permute(2,0,1).unsqueeze(0)  # [1,C,H,W]
        if pe.shape[1] != C:  # 對齊通道
            pe = F.interpolate(pe, size=(H,W), mode='bilinear', align_corners=False)
            pe = pe[:,:C]
        return x + pe

# ---- 4) GBC：Lightweight Gated Bottleneck Convolution（對齊式(4)–(8)）----
class BottConv(nn.Module):
    # 低秩瓶頸：1x1 降維 → 深度可膨脹 3x3 → 1x1 升維
    def __init__(self, c, r=4, d=1):
        super().__init__()
        mid = max(c//r, 8)
        self.reduce = conv_bn_act(c, mid, k=1, s=1, d=1)
        self.dw     = conv_bn_act(mid, mid, k=3, s=1, d=d, groups=mid)
        self.expand = conv_bn_act(mid, c,   k=1, s=1, d=1)
    def forward(self, x):
        return self.expand(self.dw(self.reduce(x)))

class GBC(nn.Module):
    # 依論文描述：先取 g1、再 BottConv→x1、再 BottConv→g2，做 Hadamard 乘積 m
    # 之後再經 BottConv + Norm + ReLU，最後加 residual（式(4)–(8)）
    def __init__(self, c, dil=2):
        super().__init__()
        self.f1 = nn.Conv2d(c, c, 1, 1, 0)  # f1(x) 只做線性投影以近似文中 f1
        self.bn1 = nn.BatchNorm2d(c)
        self.bc1 = BottConv(c, r=4, d=dil)
        self.bn2 = nn.BatchNorm2d(c)
        self.bc2 = BottConv(c, r=4, d=dil)
        self.bc3 = BottConv(c, r=4, d=1)
        self.bn3 = nn.BatchNorm2d(c)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        g1 = self.act(self.bn1(self.f1(x)))                 # (4)
        x1 = self.act(self.bn2(self.bc1(g1)))               # (5)
        g2 = self.act(self.bn2(self.bc2(x1)))               # (5) 兩個 BottConv 取特徵
        m  = g1 * g2                                        # (6) Hadamard
        y  = self.act(self.bn3(self.bc3(m)))                # (7)
        return y + x                                        # (8)

# ---- 5) SS2D：四條掃描路徑 + 視覺 Mamba（對齊 Fig.2b、式(9)–(12)）----
def snake_indices(H, W, reverse_first=False):
    # 平行蛇形掃描（兩條路徑：從上或從下開始）
    idx = []
    rows = list(range(H))
    if reverse_first: rows = rows[::-1]
    for r in rows:
        if r % 2 == 0:
            idx.extend([(r, c) for c in range(W)])
        else:
            idx.extend([(r, c) for c in range(W-1, -1, -1)])
    return idx

def diag_snake_indices(H, W, anti=False):
    # 對角蛇形（兩條路徑：主對角 or 反對角）
    idx = []
    if not anti:
        # i+j = const
        for s in range(H+W-1):
            diag = []
            for i in range(max(0, s-(W-1)), min(H-1, s)+1):
                j = s - i
                diag.append((i,j))
            if s % 2 == 0: diag = diag[::-1]
            idx.extend(diag)
    else:
        # i + (W-1-j) = const（反對角）
        for s in range(H+W-1):
            diag = []
            for i in range(max(0, s-(W-1)), min(H-1, s)+1):
                j = (W-1) - (s - i)
                diag.append((i,j))
            if s % 2 == 0: diag = diag[::-1]
            idx.extend(diag)
    return idx

class SS2D_Mamba(nn.Module):
    # 把四路掃描後的序列，各自丟進 Mamba，再還原空間排列，最後加總
    def __init__(self, c, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.BatchNorm2d(c)
        self.m1 = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)
        self.m2 = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)
        self.m3 = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)
        self.m4 = Mamba(d_model=c, d_state=d_state, d_conv=d_conv, expand=expand)

    def _scan(self, x, route_id):
        B,C,H,W = x.shape
        if route_id == 0:
            idx = snake_indices(H,W,False)
        elif route_id == 1:
            idx = snake_indices(H,W,True)
        elif route_id == 2:
            idx = diag_snake_indices(H,W,False)
        else:
            idx = diag_snake_indices(H,W,True)

        # 取序列：B 個 batch，各有 H*W 個節點，每節點 C 維度
        coords = torch.tensor(idx, device=x.device, dtype=torch.long)
        rr, cc = coords[:,0], coords[:,1]                           # [L]
        seq = x[:,:,rr,cc]                                          # [B,C,L]
        seq = seq.permute(0,2,1).contiguous()                       # [B,L,C]
        return seq, rr, cc

    def _unscan(self, seq, rr, cc, H, W):
        B,L,C = seq.shape
        y = torch.zeros((B,C,H,W), device=seq.device, dtype=seq.dtype)
        seq = seq.permute(0,2,1).contiguous()                       # [B,C,L]
        y[:,:,rr,cc] = seq
        return y

    def forward(self, x):
        x = self.norm(x)
        B,C,H,W = x.shape
        outs = []
        for k, m in enumerate([self.m1,self.m2,self.m3,self.m4]):
            seq, rr, cc = self._scan(x, k)
            yseq = m(seq)                                          # 式(10)–(12) 的離散版更新
            y = self._unscan(yseq, rr, cc, H, W)
            outs.append(y)
        return sum(outs)  # 四路加總（Fig.2b 下排）

# ---- 6) PAF + 線性歸一化殘差（Fig.2b：SS2D→PAF→Linear→GN→Linear→Res）----
class PAFusion(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Sequential(nn.Conv2d(c*2, c, 1), nn.Sigmoid())
        self.lin1 = nn.Conv2d(c, c, 1)
        self.gn   = nn.GroupNorm(num_groups= min(32, c), num_channels=c)
        self.lin2 = nn.Conv2d(c, c, 1)
    def forward(self, x, y):           # x: 原特徵；y: SS2D 後特徵
        a = self.gate(torch.cat([x,y], dim=1))
        z = a*y + (1-a)*x              # Pixel Attention Oriented Fusion（文中 PAF）
        z = self.lin2(self.gn(self.lin1(z)))
        return z + x                   # 殘差

# ---- 7) SAVSS：SS2D + PAF 殘差，再接 GBC 精修（對齊 3.3 節描述）----
class SAVSS(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ss2d = SS2D_Mamba(c)
        self.paf  = PAFusion(c)
        self.gbc  = GBC(c, dil=2)
    def forward(self, x):
        y = self.ss2d(x)
        z = self.paf(x, y)
        return self.gbc(z)

# ---- 8) DySample（ICCV'23：Learning to Upsample by Learning to Sample 的單點動態取樣）----
# 這裡實作「每個上採樣位置一個動態偏移」，用 grid_sample 完成；不需額外 CUDA。
class DySample2x(nn.Module):
    def __init__(self, c, max_off=1.0):
        super().__init__()
        self.offset = nn.Conv2d(c, 2, 3, 1, 1)   # 產生 (dx, dy)，tanh 壓到 [-1,1]
        self.max_off = max_off
    def forward(self, x):
        B,C,H,W = x.shape
        # 基礎 2x 網格（normalized [-1,1]）
        yy, xx = torch.meshgrid(
            torch.linspace(-1, 1, 2*H, device=x.device),
            torch.linspace(-1, 1, 2*W, device=x.device),
            indexing='ij'
        )
        base = torch.stack([xx, yy], dim=-1).unsqueeze(0).repeat(B,1,1,1)  # [B,2H,2W,2]
        off  = torch.tanh(self.offset(x))                                   # [B,2,H,W]? 先在原尺
        off  = F.interpolate(off, size=(2*H,2*W), mode='bilinear', align_corners=False)
        # 把像素偏移轉成 normalized 座標偏移（約略用 1/W、1/H 尺度）
        dx = (off[:,0]/W)*self.max_off
        dy = (off[:,1]/H)*self.max_off
        grid = torch.stack([base[...,0]+dx, base[...,1]+dy], dim=-1)        # [B,2H,2W,2]
        y = F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)
        return y

# ---- 9) 多尺度分割頭（對齊式(13)–(15)）----
class MLP_Head(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.mlp = nn.Sequential(nn.Conv2d(c, c, 1), nn.SiLU(inplace=True),
                                 nn.Conv2d(c, c, 1), nn.SiLU(inplace=True))
    def forward(self, x): return self.mlp(x)

class MFS_Head(nn.Module):
    # 論文式(13)：F_i^up = DySample(MLP(F_i))；再把四層 concat → GBC → Conv → MLP → Conv → logits（式(14)–(15)）
    def __init__(self, c1, c2, c3, c4, out_c=1):
        super().__init__()
        self.mlp1, self.mlp2, self.mlp3, self.mlp4 = MLP_Head(c1), MLP_Head(c2), MLP_Head(c3), MLP_Head(c4)
        self.up2 = DySample2x(c2); self.up3a = DySample2x(c3); self.up3b = DySample2x(c3)
        self.up4a = DySample2x(c4); self.up4b = DySample2x(c4); self.up4c = DySample2x(c4)
        # 將各尺度對齊到與 F1 同分辨率（c2: ×2；c3: ×4；c4: ×8）
        self.align2 = nn.Conv2d(c2, c1, 1); self.align3 = nn.Conv2d(c3, c1, 1); self.align4 = nn.Conv2d(c4, c1, 1)
        self.gbc   = GBC(c=4*c1, dil=1)
        self.conv  = conv_bn_act(4*c1, 2*c1, k=3)
        self.mlp_o = nn.Sequential(nn.Conv2d(2*c1, 2*c1, 1), nn.SiLU(inplace=True),
                                   nn.Conv2d(2*c1, c1, 1), nn.SiLU(inplace=True))
        self.head  = nn.Conv2d(c1, out_c, 1)
    def forward(self, F1, F2, F3, F4):
        F1u = self.mlp1(F1)
        F2u = self.mlp2(F2); F2u = self.up2(F2u);                 # ×2
        F3u = self.mlp3(F3); F3u = self.up3b(self.up3a(F3u));     # ×4
        F4u = self.mlp4(F4); F4u = self.up4c(self.up4b(self.up4a(F4u)))  # ×8
        F2u = self.align2(F2u); F3u = self.align3(F3u); F4u = self.align4(F4u)
        cat = torch.cat([F1u, F2u, F3u, F4u], dim=1)              # (14) Concat
        o1  = self.gbc(cat)                                       # (14) GBC
        o1  = self.conv(o1)
        o   = self.mlp_o(o1)                                      # (15) MLP(Conv(.))
        return self.head(o)                                       # logits

# ---- 10) 主幹網路（Fig.2a 整體）----
class SCSegamba(nn.Module):
    def __init__(self, base=16):  # 論文 2.8M 等級，這裡 base=16/24/32 可調
        super().__init__()
        c1, c2, c3, c4 = base, base*2, base*4, base*8
        self.embed = conv_bn_act(3, c1, k=3)
        self.pe    = PosEnc2D(c1)
        self.b1 = SAVSS(c1)
        self.e2 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c1, c2, k=3))
        self.b2 = SAVSS(c2)
        self.e3 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c2, c3, k=3))
        self.b3 = SAVSS(c3)
        self.e4 = nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c3, c4, k=3))
        self.b4 = SAVSS(c4)
        self.head = MFS_Head(c1, c2, c3, c4, out_c=1)
        # 參數統計
        self._n_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
    def forward(self, x):
        f1 = self.pe(self.embed(x)); f1 = self.b1(f1)
        f2 = self.b2(self.e2(f1))
        f3 = self.b3(self.e3(f2))
        f4 = self.b4(self.e4(f3))
        return self.head(f1,f2,f3,f4)  # logits

# ---- 11) 損失：BCE + Dice，α:β = 1:5（式(16)）----
class BCEDiceLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=5.0, eps=1e-7):
        super().__init__()
        self.alpha, self.beta, self.eps = alpha, beta, eps
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, y):
        bce = self.bce(logits, y)
        p   = torch.sigmoid(logits)
        inter = (p*y).sum(dim=(1,2,3))
        summ  = (p+y).sum(dim=(1,2,3))
        dice  = 1 - (2*inter + self.eps) / (summ + self.eps)
        dice  = dice.mean()
        return self.alpha*bce + self.beta*dice

@torch.no_grad()
def seg_metrics(logits, y, thr=0.5, eps=1e-7):
    p = (torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou = tp/(tp+fp+fn+eps); f1= 2*tp/(2*tp+fp+fn+eps)
    return iou, f1

# ---- 12) 訓練/驗證 迴圈 ----
def train(
    data_root="/content/TUT", work_dir="/content/work_scsegamba",
    base=16, crop=512, batch=4, epochs=40, lr=3e-4, amp=True
):
    Path(work_dir).mkdir(parents=True, exist_ok=True)
    device="cuda" if torch.cuda.is_available() else "cpu"
    tr = CrackSegSet(data_root,"train",crop); va=CrackSegSet(data_root,"val",crop)
    tl = DataLoader(tr,batch,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
    vl = DataLoader(va,1,shuffle=False,num_workers=2)

    model = SCSegamba(base=base).to(device)
    print(f"Model params: {model._n_params/1e6:.2f}M")
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs,1))
    loss_fn = BCEDiceLoss(alpha=1.0, beta=5.0)

    use_amp = amp and (device=="cuda")
    scaler = torch.amp.GradScaler('cuda') if use_amp else None

    best_iou=-1; best = str(Path(work_dir)/"scsegamba_best.pth")
    for ep in range(1,epochs+1):
        model.train(); t0=time.time()
        for x,y in tl:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=use_amp):
                logits = model(x)
                loss   = loss_fn(logits,y)
            if use_amp:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
        sch.step()
        # valid
        model.eval(); ious=[]; f1s=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits=model(x)
                iou,f1=seg_metrics(logits,y); ious.append(iou); f1s.append(f1)
        miou=float(np.mean(ious)); mf1=float(np.mean(f1s))
        print(f"Epoch {ep:03d}/{epochs} | val mIoU={miou:.4f}  F1={mf1:.4f}  time={time.time()-t0:.1f}s")
        if miou>best_iou:
            best_iou=miou; torch.save(model.state_dict(), best)
    print(f"[Done] best mIoU={best_iou:.4f} | saved to {best}")
    return model

# ---- 13) 啟動 ----
DATA_ROOT = "/content/TUT"  # <<< 改成你的資料夾（TUT/DeepCrack/CrackMap 皆可，只要照資料夾結構）
assert Path(DATA_ROOT).exists(), f"未找到 {DATA_ROOT}；請放好 /{DATA_ROOT}/{{train,val}}/{{images,masks}}/*.png"
model = train(DATA_ROOT, base=16, crop=512, batch=4, epochs=40, lr=3e-4, amp=True)


ValueError: num_samples should be a positive integer value, but got num_samples=0

# 323

In [ ]:
# =================== SCSegamba — paper-aligned (stable final, NaN-proof, FIX BOOL) ===================
# 論文對齊：GBC、SS2D(四路+Mamba)、PAF、MFS(DySample)
# 損失：Loss = Dice + 5*BCE（與論文一致；BCE 負責分離、Dice 做幾何微調）
# 穩定化：SS2D_Mamba 強制 FP32、logits/loss 有限性檢查、DySample grid 夾住、val 單工讀取
# 注意：把所有「not tensor」改成「not tensor.all().item()」，避免布林歧義錯誤
# =====================================================================================================

# ---- 0) 依賴 & Mamba 後備 ----
# !pip -q install albumentations==1.4.7 opencv-python==4.10.0.84

import importlib, os, time, random, math
from pathlib import Path
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

SEED=2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MAMBA_AVAILABLE=True
try:
    importlib.import_module("mamba_ssm")
except:
    MAMBA_AVAILABLE=False
    !pip -q install "mamba-ssm[causal-conv1d]>=2.2.4" || true
    try:
        importlib.import_module("mamba_ssm"); MAMBA_AVAILABLE=True
    except:
        MAMBA_AVAILABLE=False

if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba
else:
    print("[warn] mamba-ssm 不可用，改用 GRU 後備（可跑、較慢）")
    class Mamba(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__(); self.rnn=nn.GRU(d_model,d_model,batch_first=True)
        def forward(self,x): y,_=self.rnn(x); return y

# ---- 1) 資料偵測 ----
IMG_EXT={".png",".jpg",".jpeg",".bmp",".webp"}
IMAGE_DIR_CAND=["images","image","img","imgs","JPEGImages"]
MASK_DIR_CAND=["masks","mask","labels","label","ann","anno","annotations","gt","groundtruth","ground-truth"]
SPLIT_CAND=["train","val","valid","validation","test"]

def _find_first_subdir(base: Path, cands):
    for n in cands:
        p=base/n
        if p.exists() and any(p.glob("*")): return p
    return None

def detect_or_prepare_dataset(root="/content/TUT"):
    root=Path(root); root.mkdir(parents=True, exist_ok=True)
    found={}
    for sp in SPLIT_CAND:
        d=root/sp
        if d.exists():
            img=_find_first_subdir(d, IMAGE_DIR_CAND)
            msk=_find_first_subdir(d, MASK_DIR_CAND)
            if img and msk: found[sp]=(img,msk)
    if ("train" in found) and any(k in found for k in ["val","valid","validation"]):
        if "val" not in found:
            found["val"]=found.get("validation", found.get("valid"))
        print(f"[info] 偵測到現有 train/val 結構：{root}")
        return str(root)

    img_dir=_find_first_subdir(root, IMAGE_DIR_CAND)
    msk_dir=_find_first_subdir(root, MASK_DIR_CAND)
    if img_dir and msk_dir:
        print(f"[info] 偵測到根目錄 images+masks，建立 train/val：{root}")
        imgs=sorted([p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT]); assert len(imgs)>0
        for sp in ["train","val"]:
            (root/sp/"images").mkdir(parents=True, exist_ok=True)
            (root/sp/"masks").mkdir(parents=True, exist_ok=True)
        n=len(imgs); nv=max(1,int(n*0.1)); valset=set(imgs[-nv:])
        for p in imgs:
            q=msk_dir/p.name
            if not q.exists():
                q_alt=None
                for ext in IMG_EXT:
                    cand=msk_dir/(p.stem+ext)
                    if cand.exists(): q_alt=cand; break
                if q_alt is None: continue
                q=q_alt
            dst="val" if p in valset else "train"
            di=root/dst/"images"/p.name; dm=root/dst/"masks"/q.name
            if not di.exists(): os.link(p, di)
            if not dm.exists(): os.link(q, dm)
        return str(root)

    print("[info] 未偵測到資料；合成迷你資料（僅 smoke test）…")
    def synth(H=512,W=512):
        img=(np.random.rand(H,W,3)*40+100).astype(np.uint8); img=cv2.GaussianBlur(img,(0,0),1.5)
        m=np.zeros((H,W),np.uint8)
        for _ in range(np.random.randint(1,3)):
            x1,y1=np.random.randint(0,W),np.random.randint(0,H)
            x2,y2=np.random.randint(0,W),np.random.randint(0,H)
            cv2.line(m,(x1,y1),(x2,y2),255,thickness=np.random.randint(1,3))
        m=cv2.GaussianBlur(m,(3,3),0); m=(m>127).astype(np.uint8)*255
        return img,m
    for sp,n in [("train",24),("val",6)]:
        (root/sp/"images").mkdir(parents=True, exist_ok=True)
        (root/sp/"masks").mkdir(parents=True, exist_ok=True)
        for i in range(n):
            x,y=synth(512,512)
            cv2.imwrite(str(root/sp/"images"/f"{i:05d}.png"), cv2.cvtColor(x,cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(root/sp/"masks"/f"{i:05d}.png"), y)
    print("[info] 合成完成：", root)
    return str(root)

# ---- 2) Dataset ----
def _mask_to01(a, invert=False):
    a=(a>127).astype(np.float32)
    return 1.0-a if invert else a

def _ensure_multiple_of_8(h,w):
    H=(h//8)*8; W=(w//8)*8
    if H==0: H=8
    if W==0: W=8
    return H,W

def _center_crop_to(h,w,ch,cw,cx,cy):
    top=max(0, min(h-ch, cy-ch//2)); left=max(0, min(w-cw, cx-cw//2))
    return int(top), int(left)

class CrackSegSet(Dataset):
    def __init__(self, root, split="train", crop=256, auto_invert=True, pos_center=True, tries=30):
        root=Path(root)/split
        img_dir=_find_first_subdir(root, IMAGE_DIR_CAND)
        msk_dir=_find_first_subdir(root, MASK_DIR_CAND)
        assert img_dir and msk_dir, f"{root} 缺 images 或 masks/labels"
        self.X=sorted([str(p) for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y=[]
        for p in self.X:
            q=msk_dir/Path(p).name
            if not q.exists():
                for ext in IMG_EXT:
                    cand=msk_dir/(Path(p).stem+ext)
                    if cand.exists(): q=cand; break
            if q.exists(): self.Y.append(str(q))
        self.split=split; self.crop=crop; self.pos_center=(pos_center and split=="train"); self.tries=tries

        self.tf=A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.95,1.05), rotate=(-8,8), translate_percent=(0.0,0.03), p=0.5),
            A.RandomBrightnessContrast(0.08,0.08,p=0.3),
        ])

        self._invert=False; pos=0.0; cnt=0
        self._has_pos_idx=[]
        for i in range(len(self.X)):
            y=np.array(Image.open(self.Y[i]).convert("L"))
            m=(y>127).astype(np.uint8); r=m.mean()
            if r>0: self._has_pos_idx.append(i)
            if cnt<32: pos+=r; cnt+=1
        est= pos/cnt if cnt>0 else 0.0
        if auto_invert and est>0.5: self._invert=True
        print(f"[{split}] samples={len(self.X)}  est_pos_ratio={est:.4f}  invert={self._invert}")

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        if self.split=="train" and self.pos_center and self._has_pos_idx:
            if random.random()<0.8:
                i=random.choice(self._has_pos_idx)

        x=np.array(Image.open(self.X[i]).convert("RGB"))
        y=np.array(Image.open(self.Y[i]).convert("L"))
        y01=_mask_to01(y, invert=self._invert)

        H,W=y01.shape

        if self.split=="train":
            ch,cw=self.crop,self.crop
            if H<ch or W<cw:
                ph=max(0,ch-H); pw=max(0,cw-W)
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y01=cv2.copyMakeBorder(y01,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                H,W=y01.shape
            if self.pos_center and (y01.sum()>0):
                ys,xs=np.where(y01>0.5)
                cy=int(np.median(ys)); cx=int(np.median(xs))
                top,left=_center_crop_to(H,W,ch,cw,cx,cy)
            else:
                top=random.randint(0,H-ch); left=random.randint(0,W-cw)
            xc=x[top:top+ch, left:left+cw]; yc=y01[top:top+ch, left:left+cw]
            aug=self.tf(image=xc, mask=(yc*255).astype(np.uint8))
            x=aug["image"]; y01=(aug["mask"]>127).astype(np.float32)

        else:
            H2,W2=_ensure_multiple_of_8(H,W)
            if H2!=H or W2!=W:
                top=(H-H2)//2; left=(W-W2)//2
                x=x[top:top+H2, left:left+W2]
                y01=y01[top:top+H2, left:left+W2]
                H,W=H2,W2
            ch,cw=self.crop,self.crop
            if H>=ch and W>=cw:
                if y01.sum()>0:
                    ys,xs=np.where(y01>0.5)
                    cy=int(np.median(ys)); cx=int(np.median(xs))
                    top,left=_center_crop_to(H,W,ch,cw,cx,cy)
                else:
                    top=(H-ch)//2; left=(W-cw)//2
                x=x[top:top+ch, left:left+cw]; y01=y01[top:top+ch, left:left+cw]
            else:
                ph=max(0,ch-H); pw=max(0,cw-W)
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y01=cv2.copyMakeBorder(y01,0,ph,0,pw,cv2.BORDER_REFLECT_101)

        x=torch.from_numpy(x.transpose(2,0,1)).float()/255.0
        y=torch.from_numpy(y01[None]).float()
        return x,y

# ---- 3) 模型（SS2D_Mamba 強制 FP32）----
def conv_bn_act(cin, cout, k=3, s=1, d=1, groups=1, act=nn.SiLU):
    pad=((k-1)//2)*d
    return nn.Sequential(nn.Conv2d(cin,cout,k,s,pad,dilation=d,groups=groups,bias=False),
                         nn.BatchNorm2d(cout), act(inplace=True))

class PosEnc2D(nn.Module):
    def __init__(self,C,temperature=10000.0):
        super().__init__(); self.C=C; self.temperature=temperature
    def forward(self,x):
        B,C,H,W=x.shape
        y,xg=torch.meshgrid(torch.arange(H,device=x.device), torch.arange(W,device=x.device), indexing='ij')
        n=max(C//4,1); denom=max(C//2,1)
        omega=torch.arange(n,device=x.device).float()
        omega=1.0/(self.temperature**(2*omega/denom))
        y=y[...,None]*omega; xg=xg[...,None]*omega
        pe=torch.cat([torch.sin(xg),torch.cos(xg),torch.sin(y),torch.cos(y)],-1)
        pe=pe.permute(2,0,1).unsqueeze(0)
        if pe.shape[1]!=C:
            if pe.shape[1]<C: pe=torch.cat([pe, torch.zeros((1,C-pe.shape[1],H,W),device=x.device)],1)
            else: pe=pe[:,:C]
        return x+pe

class BottConv(nn.Module):
    def __init__(self,c,r=4,d=1):
        super().__init__()
        mid=max(c//r,8)
        self.reduce=conv_bn_act(c,mid,1,1,1)
        self.dw    =conv_bn_act(mid,mid,3,1,d,groups=mid)
        self.expand=conv_bn_act(mid,c,1,1,1)
    def forward(self,x): return self.expand(self.dw(self.reduce(x)))

class GBC(nn.Module):
    def __init__(self,c,dil=2):
        super().__init__()
        self.f1=nn.Conv2d(c,c,1); self.bn1=nn.BatchNorm2d(c)
        self.bc1=BottConv(c,4,dil); self.bn2=nn.BatchNorm2d(c)
        self.bc2=BottConv(c,4,dil); self.bc3=BottConv(c,4,1); self.bn3=nn.BatchNorm2d(c)
        self.act=nn.ReLU(inplace=True)
    def forward(self,x):
        g1=self.act(self.bn1(self.f1(x)))
        x1=self.act(self.bn2(self.bc1(g1)))
        g2=self.act(self.bn2(self.bc2(x1)))
        m=g1*g2
        y=self.act(self.bn3(self.bc3(m)))
        return y+x

def _snake_pairs(H,W,rev=False):
    rows=list(range(H))[::-1] if rev else list(range(H))
    idx=[]
    for r in rows:
        if r%2==0: idx.extend([(r,c) for c in range(W)])
        else: idx.extend([(r,c) for c in range(W-1,-1,-1)])
    return idx

def _diag_pairs(H,W,anti=False):
    idx=[]
    if not anti:
        for s in range(H+W-1):
            d=[]
            for i in range(max(0,s-(W-1)), min(H-1,s)+1):
                j=s-i; d.append((i,j))
            if s%2==0: d=d[::-1]
            idx.extend(d)
    else:
        for s in range(H+W-1):
            d=[]
            for i in range(max(0,s-(W-1)), min(H-1,s)+1):
                j=(W-1)-(s-i); d.append((i,j))
            if s%2==0: d=d[::-1]
            idx.extend(d)
    return idx

class SS2D_Mamba(nn.Module):
    def __init__(self,c,d_state=16,d_conv=4,expand=2):
        super().__init__()
        self.norm=nn.BatchNorm2d(c)
        self.m1=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m2=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m3=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m4=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self._cache={}
    def _rrcc(self,H,W,route,device):
        k=(H,W,route)
        if k not in self._cache:
            if route==0: pairs=_snake_pairs(H,W,False)
            elif route==1: pairs=_snake_pairs(H,W,True)
            elif route==2: pairs=_diag_pairs(H,W,False)
            else: pairs=_diag_pairs(H,W,True)
            arr=np.array(pairs, dtype=np.int64)
            self._cache[k]=(torch.from_numpy(arr[:,0]), torch.from_numpy(arr[:,1]))
        rr,cc=self._cache[k]
        return rr.to(device), cc.to(device)
    def forward(self,x):
        # ★ 關閉 AMP、強制 FP32（用新的 torch.amp.autocast 介面）
        with torch.amp.autocast('cuda', enabled=False):
            x32 = x.to(torch.float32)
            x32 = self.norm(x32)
            B,C,H,W=x32.shape; outs=[]
            for k,m in enumerate([self.m1,self.m2,self.m3,self.m4]):
                rr,cc=self._rrcc(H,W,k,x32.device)
                seq=x32[:,:,rr,cc].permute(0,2,1).contiguous()   # [B, L, C]
                yseq=m(seq)                                      # [B, L, C]
                y=torch.zeros((B,C,H,W), device=x32.device, dtype=yseq.dtype)
                y[:,:,rr,cc]=yseq.permute(0,2,1).contiguous()
                outs.append(y)
            y_sum = sum(outs)                                    # FP32
        return y_sum.to(dtype=x.dtype)

class PAFusion(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.gate=nn.Sequential(nn.Conv2d(c*2,c,1), nn.Sigmoid())
        self.lin1=nn.Conv2d(c,c,1); self.gn=nn.GroupNorm(min(32,c),c); self.lin2=nn.Conv2d(c,c,1)
    def forward(self,x,y):
        a=self.gate(torch.cat([x,y],1))
        z=a*y+(1-a)*x
        z=self.lin2(self.gn(self.lin1(z)))
        return z+x

class SAVSS(nn.Module):
    def __init__(self,c):
        super().__init__()
        # 與論文一致：GBC → SS2D → PAF
        self.pre_gbc=GBC(c,2)
        self.ss2d=SS2D_Mamba(c)
        self.paf=PAFusion(c)
    def forward(self,x):
        x1=self.pre_gbc(x)
        y=self.ss2d(x1)
        z=self.paf(x1,y)
        return z

class DySample2x(nn.Module):
    def __init__(self,c,max_off=1.0):
        super().__init__()
        self.offset=nn.Conv2d(c,2,3,1,1)
        nn.init.zeros_(self.offset.weight)
        nn.init.zeros_(self.offset.bias)
        self.max_off=max_off; self._grid={}
    def _base(self,H,W,device):
        k=(H,W,str(device))
        if k not in self._grid:
            yy,xx=torch.meshgrid(torch.linspace(-1,1,2*H,device=device),
                                 torch.linspace(-1,1,2*W,device=device), indexing='ij')
            self._grid[k]=torch.stack([xx,yy],-1).unsqueeze(0)   # [1,2H,2W,2]
        return self._grid[k]
    def forward(self,x):
        B,C,H,W=x.shape
        base=self._base(H,W,x.device).repeat(B,1,1,1)            # [B,2H,2W,2]
        off=torch.tanh(self.offset(x))                           # [B,2,H,W] before up
        off=F.interpolate(off,size=(2*H,2*W),mode='bilinear',align_corners=False)
        dx=(off[:,0]/W)*self.max_off; dy=(off[:,1]/H)*self.max_off
        grid=torch.stack([base[...,0]+dx, base[...,1]+dy],-1)    # [B,2H,2W,2]
        grid = grid.clamp(-1.0, 1.0)                             # ★ 夾住避免越界
        return F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)

class MLP_Head(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.mlp=nn.Sequential(nn.Conv2d(c,c,1), nn.SiLU(True), nn.Conv2d(c,c,1), nn.SiLU(True))
    def forward(self,x): return self.mlp(x)

class MFS_Head(nn.Module):
    def __init__(self,c1,c2,c3,c4,out_c=1):
        super().__init__()
        self.mlp1,self.mlp2,self.mlp3,self.mlp4 = MLP_Head(c1),MLP_Head(c2),MLP_Head(c3),MLP_Head(c4)
        self.up2=DySample2x(c2); self.up3a=DySample2x(c3); self.up3b=DySample2x(c3)
        self.up4a=DySample2x(c4); self.up4b=DySample2x(c4); self.up4c=DySample2x(c4)
        self.a2=nn.Conv2d(c2,c1,1); self.a3=nn.Conv2d(c3,c1,1); self.a4=nn.Conv2d(c4,c1,1)
        self.gbc=GBC(4*c1,1); self.conv=conv_bn_act(4*c1,2*c1,3)
        self.mlp_o=nn.Sequential(nn.Conv2d(2*c1,2*c1,1), nn.SiLU(True), nn.Conv2d(2*c1,c1,1), nn.SiLU(True))
        self.head=nn.Conv2d(c1,out_c,1)
    def forward(self,F1,F2,F3,F4):
        F1u=self.mlp1(F1)
        F2u=self.a2(self.up2(self.mlp2(F2)))
        F3u=self.a3(self.up3b(self.up3a(self.mlp3(F3))))
        F4u=self.a4(self.up4c(self.up4b(self.up4a(self.mlp4(F4)))))
        cat=torch.cat([F1u,F2u,F3u,F4u],1)
        o1=self.conv(self.gbc(cat))
        o =self.mlp_o(o1)
        return self.head(o)

class SCSegamba(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        c1,c2,c3,c4=base,base*2,base*4,base*8
        self.embed=conv_bn_act(3,c1,3); self.pe=PosEnc2D(c1); self.b1=SAVSS(c1)
        self.e2=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c1,c2,3)); self.b2=SAVSS(c2)
        self.e3=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c2,c3,3)); self.b3=SAVSS(c3)
        self.e4=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c3,c4,3)); self.b4=SAVSS(c4)
        self.head=MFS_Head(c1,c2,c3,c4,1)
        self._n_params=sum(p.numel() for p in self.parameters() if p.requires_grad)
    def forward(self,x):
        f1=self.b1(self.pe(self.embed(x)))
        f2=self.b2(self.e2(f1))
        f3=self.b3(self.e3(f2))
        f4=self.b4(self.e4(f3))
        return self.head(f1,f2,f3,f4)

# ---- 4) Loss / Metrics / 可視化 ----
class BCEDiceLoss(nn.Module):
    """L = α·Dice + β·BCE，預設 α=1, β=5（Dice : BCE = 1 : 5）"""
    def __init__(self, alpha=1.0, beta=5.0, pos_weight=None, eps=1e-7):
        super().__init__(); self.alpha=alpha; self.beta=beta; self.eps=eps
        self.bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    def forward(self,logits,y):
        bce=self.bce(logits,y)
        p=torch.sigmoid(logits)
        inter=(p*y).sum((1,2,3)); summ=(p+y).sum((1,2,3))
        dice_loss=1-(2*inter+self.eps)/(summ+self.eps)
        dice_loss=torch.nan_to_num(dice_loss, nan=0.0, posinf=1.0, neginf=0.0).mean()
        loss = self.alpha*dice_loss + self.beta*bce
        return loss, bce.detach(), dice_loss.detach()

@torch.no_grad()
def metrics_at(logits,y,thr=0.5,eps=1e-7):
    p=(torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou=tp/(tp+fp+fn+eps); f1=2*tp/(2*tp+fp+fn+eps)
    return iou,f1

@torch.no_grad()
def as_rgb(a: np.ndarray, to_size=None) -> np.ndarray:
    a = np.nan_to_num(a, nan=0.0, posinf=255.0, neginf=0.0).astype(np.uint8)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=-1)
    elif a.ndim == 3 and a.shape[2] == 1:
        a = np.repeat(a, 3, axis=-1)
    elif a.ndim == 3 and a.shape[2] >= 3:
        a = a[..., :3]
    else:
        a = np.repeat(a[..., None], 3, axis=-1)
    if to_size is not None:
        a = cv2.resize(a, (to_size[1], to_size[0]), interpolation=cv2.INTER_NEAREST)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=-1)
    return a

@torch.no_grad()
def save_viz(model, ds, out_dir, k=6):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    device=next(model.parameters()).device
    model.eval()
    n=min(k,len(ds))
    for i in range(n):
        x,y=ds[i]; x=x.unsqueeze(0).to(device); y=y.to(device)
        logits=model(x)
        if not torch.isfinite(logits).all().item():
            print("[viz] 非有限 logits，略過可視化");
            continue
        prob=torch.sigmoid(logits)
        img=(x[0].permute(1,2,0).cpu().numpy()*255.0)
        prb=(prob[0,0].clamp(0,1).cpu().numpy()*255.0)
        msk=(y[0,0].clamp(0,1).cpu().numpy()*255.0)
        H,W=img.shape[:2]
        img3=as_rgb(img, to_size=(H,W))
        prb3=as_rgb(prb, to_size=(H,W))
        msk3=as_rgb(msk, to_size=(H,W))
        vis=np.vstack([img3, prb3, msk3])
        cv2.imwrite(str(Path(out_dir)/f"viz_{i:02d}.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))

# ---- 5) 先驗與頭部偏置 ----
def estimate_pos_ratio_full(ds):
    s=0.0; c=0
    for i in range(len(ds.X)):
        y=np.array(Image.open(ds.Y[i]).convert("L"))
        s+=(y>127).mean(); c+=1
    return (s/c) if c>0 else 0.01

def set_head_prior_bias(model, p_prior=0.01):
    p=float(np.clip(p_prior, 1e-4, 0.1))
    b=math.log(p/(1-p))
    for m in reversed(list(model.modules())):
        if isinstance(m, nn.Conv2d) and m.out_channels==1 and m.bias is not None:
            m.bias.data.fill_(b); print(f"[info] head bias set to logit({p:.4f}) = {b:.3f}")
            break

# ---- 6) 訓練 ----
def train(
    data_root="/content/TUT", work_dir="/content/work_scsegamba",
    base=16, crop=256, batch=6, epochs=30, lr=5e-4, amp=False  # ★ 先關 AMP，穩了再開
):
    Path(work_dir).mkdir(parents=True, exist_ok=True)
    device="cuda" if torch.cuda.is_available() else "cpu"
    pin = (device=="cuda")
    data_root=detect_or_prepare_dataset(data_root)

    tr=CrackSegSet(data_root,"train",crop,auto_invert=True,pos_center=True,tries=30)
    va=CrackSegSet(data_root,"val",  crop,auto_invert=True,pos_center=False)

    tl=DataLoader(tr,batch,shuffle=True,num_workers=(2 if pin else 0),pin_memory=pin,drop_last=True)
    vl=DataLoader(va,1,shuffle=False,num_workers=0,pin_memory=pin)   # ★ val 單工

    p_prior=estimate_pos_ratio_full(tr)
    p_prior=max(min(p_prior,0.49),1e-4)
    pos_w=float(np.clip((1-p_prior)/p_prior, 1.0, 20.0))
    print(f"[info] 整體先驗正率 p≈{p_prior:.4f} → pos_weight={pos_w:.2f}")
    model=SCSegamba(base=base).to(device).float()
    set_head_prior_bias(model, p_prior=p_prior)

    print(f"Model params: {model._n_params/1e6:.2f}M | Mamba: {MAMBA_AVAILABLE}")
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs,1))
    loss_fn=BCEDiceLoss(alpha=1.0, beta=5.0, pos_weight=torch.tensor([pos_w], device=device))

    use_amp=amp and (device=="cuda")
    scaler=torch.amp.GradScaler('cuda') if use_amp else None

    best_iou=-1; best=str(Path(work_dir)/"scsegamba_best.pth")
    nan_batches = 0

    for ep in range(1,epochs+1):
        model.train(); t0=time.time()
        run_loss=run_bce=run_dice=0.0; nstep=0; logit_mean=0.0

        for x,y in tl:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast('cuda', enabled=True):
                    logits=model(x)
                if not torch.isfinite(logits).all().item():
                    nan_batches += 1; print("[warn] skip non-finite logits (AMP)"); continue
                loss,bce,dice=loss_fn(logits,y)
                if not torch.isfinite(loss).all().item():
                    nan_batches += 1; print("[warn] skip NaN/Inf loss (AMP)"); continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                logits=model(x)
                if not torch.isfinite(logits).all().item():
                    nan_batches += 1; print("[warn] skip non-finite logits"); continue
                loss,bce,dice=loss_fn(logits,y)
                if not torch.isfinite(loss).all().item():
                    nan_batches += 1; print("[warn] skip NaN/Inf loss"); continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            nstep+=1
            run_loss+=loss.detach().item()
            run_bce+=bce.item()
            run_dice+=dice.item()
            logit_mean+=logits.mean().detach().item()

        sch.step()

        # ---- 驗證 ----
        model.eval(); iou3=[]; f13=[]; iou5=[]; f15=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits=model(x)
                if not torch.isfinite(logits).all().item():
                    print("[val] 非有限 logits，略過該樣本");
                    continue
                a,b=metrics_at(logits,y,thr=0.3); iou3.append(a); f13.append(b)
                a,b=metrics_at(logits,y,thr=0.5); iou5.append(a); f15.append(b)
        miou3=float(np.mean(iou3)) if iou3 else 0.0
        mf13 =float(np.mean(f13)) if f13 else 0.0
        miou5=float(np.mean(iou5)) if iou5 else 0.0
        mf15 =float(np.mean(f15)) if f15 else 0.0

        print(f"Ep {ep:03d}/{epochs} | "
              f"Train loss={(run_loss/max(nstep,1)):.4f} (BCE={(run_bce/max(nstep,1)):.4f}, Dice={(run_dice/max(nstep,1)):.4f}) "
              f"logit_mean={(logit_mean/max(nstep,1)):.3f} | "
              f"mIoU@0.3={miou3:.4f} F1@0.3={mf13:.4f} | mIoU@0.5={miou5:.4f} F1@0.5={mf15:.4f} | "
              f"{time.time()-t0:.1f}s  | skipped={nan_batches}")

        if ep%5==0:
            save_viz(model, va, Path(work_dir)/f"viz_ep{ep}", k=min(6,len(va)))

        if miou5>best_iou:
            best_iou=miou5; torch.save(model.state_dict(), best)

    print(f"[Done] best mIoU@0.5={best_iou:.4f} | saved to {best}")
    return model

# ---- 7) 執行 ----
DATA_ROOT="/content/TUT"
model=train(DATA_ROOT, base=16, crop=256, batch=3, epochs=100, lr=5e-4, amp=False)  # 先穩定


[info] 偵測到現有 train/val 結構：/content/TUT
[train] samples=24  est_pos_ratio=0.0029  invert=False
[val] samples=6  est_pos_ratio=0.0022  invert=False
[info] 整體先驗正率 p≈0.0029 → pos_weight=20.00
[info] head bias set to logit(0.0029) = -5.830
Model params: 0.99M | Mamba: True
Ep 001/100 | Train loss=6.8153 (BCE=1.1638, Dice=0.9962) logit_mean=-5.817 | mIoU@0.3=0.0000 F1@0.3=0.0000 | mIoU@0.5=0.0000 F1@0.5=0.0000 | 2.4s  | skipped=0
Ep 002/100 | Train loss=5.7614 (BCE=0.9530, Dice=0.9963) logit_mean=-5.799 | mIoU@0.3=0.0000 F1@0.3=0.0000 | mIoU@0.5=0.0000 F1@0.5=0.0000 | 2.0s  | skipped=0
Ep 003/100 | Train loss=6.4504 (BCE=1.0909, Dice=0.9961) logit_mean=-5.778 | mIoU@0.3=0.0000 F1@0.3=0.0000 | mIoU@0.5=0.0000 F1@0.5=0.0000 | 2.0s  | skipped=0
Ep 004/100 | Train loss=6.4368 (BCE=1.0882, Dice=0.9959) logit_mean=-5.756 | mIoU@0.3=0.0000 F1@0.3=0.0000 | mIoU@0.5=0.0000 F1@0.5=0.0000 | 1.9s  | skipped=0
Ep 005/100 | Train loss=5.6166 (BCE=0.9241, Dice=0.9960) logit_mean=-5.731 | mIoU@0.3=0.0000 F1

In [ ]:
# ============================================
# SCSegamba 推論（滑窗+TTA+加權+後處理+進度條）
# ============================================
import os, math, cv2, torch, numpy as np
from pathlib import Path
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- 1) 載入模型與權重 ----
ckpt_path = "/content/work_scsegamba/scsegamba_best.pth"
model = SCSegamba(base=16).to(device).eval()
state = torch.load(ckpt_path, map_location=device)
model.load_state_dict(state, strict=True)

# ---- 2) 小工具 ----
def to_chw(img_rgb):
    # img_rgb: uint8 [H,W,3] -> float tensor [1,3,H,W] in [0,1]
    x = torch.from_numpy(img_rgb.transpose(2,0,1)).float()/255.0
    return x.unsqueeze(0)

def colorize(prob, cmap=cv2.COLORMAP_JET):
    pm=(np.clip(prob,0,1)*255).astype(np.uint8)
    return cv2.applyColorMap(pm, cmap)[:,:,::-1]  # BGR->RGB

def resize_like(img, size_hw, is_mask=False):
    H,W = size_hw
    interp = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    if img.ndim==2:
        return cv2.resize(img, (W,H), interpolation=interp)
    else:
        return cv2.resize(img, (W,H), interpolation=interp)

def make_hann2d(h, w, device):
    # 2D 漢寧窗，避免滑窗拼接邊界突兀
    wy = np.hanning(h); wx = np.hanning(w)
    w2 = np.outer(wy, wx)
    # 讓中心權重大、邊界權重小，但最低不為 0 以避免除以 0
    w2 = w2 / (w2.max()+1e-8)
    w2 = np.clip(w2, 1e-3, 1.0)
    return torch.from_numpy(w2).to(device).float()[None,None]  # [1,1,h,w]

@torch.no_grad()
def _tta_logits(m, x):
    y  = m(x)
    y += torch.flip(m(torch.flip(x, [-1])), [-1])
    y += torch.flip(m(torch.flip(x, [-2])), [-2])
    y += torch.flip(m(torch.flip(x, [-2, -1])), [-2, -1])
    return y / 4.0

@torch.no_grad()
def slide_predict(m, img_rgb, tile=256, overlap=0.5, use_tta=True):
    """
    img_rgb: uint8 RGB [H,W,3]
    return: prob [H,W] float32 in [0,1]
    """
    H, W = img_rgb.shape[:2]
    stride = max(1, int(tile*(1-overlap)))
    x = to_chw(img_rgb).to(device)                # [1,3,H,W]

    if H<=tile and W<=tile:
        logits = _tta_logits(m, x) if use_tta else m(x)
        return torch.sigmoid(logits)[0,0].float().cpu().numpy()

    # 反射填補到能被滑窗覆蓋
    pad_h = (math.ceil((H - tile)/stride)*stride + tile - H) if H>tile else (tile-H)
    pad_w = (math.ceil((W - tile)/stride)*stride + tile - W) if W>tile else (tile-W)
    xpad = torch.nn.functional.pad(x, (0,pad_w,0,pad_h), mode='reflect')
    Hp, Wp = int(xpad.shape[-2]), int(xpad.shape[-1])

    out = torch.zeros((1,1,Hp,Wp), device=device)
    cnt = torch.zeros((1,1,Hp,Wp), device=device)
    wwin = make_hann2d(tile, tile, device=device)

    for top in range(0, Hp - tile + 1, stride):
        for left in range(0, Wp - tile + 1, stride):
            crop = xpad[:,:,top:top+tile, left:left+tile]
            logits = _tta_logits(m, crop) if use_tta else m(crop)
            prob = torch.sigmoid(logits)
            out[:,:,top:top+tile,left:left+tile] += prob * wwin
            cnt[:,:,top:top+tile,left:left+tile] += wwin

    out = out / torch.clamp(cnt, min=1e-6)
    out = out[:,:,:H,:W]
    return out[0,0].float().cpu().numpy()  # [H,W]

# ---- 3) 後處理（讓結果接近「細線 + 去雜點」）----
def post_process(prob, thr=0.85, remove_small=80, do_skeleton=True):
    """
    prob: [H,W] in [0,1]
    thr:  閾值（建議用你 val 掃到的最佳 F1 閾值）
    remove_small: 連通區域像素數下限，小於此值會被移除
    do_skeleton: 是否做骨架化（更接近單像素寬）
    """
    binm = (prob >= float(thr)).astype(np.uint8) * 255

    # 移除小連通元件
    num, lab, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
    keep = np.zeros_like(binm)
    for i in range(1, num):
        if stats[i, cv2.CC_STAT_AREA] >= int(remove_small):
            keep[lab==i] = 255
    binm = keep

    # 輕微平滑與細化
    binm = cv2.morphologyEx(binm, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_RECT,(2,2)),
                            iterations=1)

    if do_skeleton:
        # 盡量單像素寬（OpenCV 無 ximgproc 時用形態骨架）
        skel = np.zeros_like(binm)
        element = cv2.getStructuringElement(cv2.MORPH_CROSS, (3,3))
        temp = np.zeros_like(binm)
        eroded = np.copy(binm)
        while True:
            eroded = cv2.erode(eroded, element)
            temp = cv2.dilate(eroded, element)
            temp = cv2.subtract(binm, temp)
            skel = cv2.bitwise_or(skel, temp)
            binm = eroded.copy()
            if cv2.countNonZero(binm) == 0:
                break
        binm = (skel>0).astype(np.uint8)*255

        # 再移除超短枝幹
        num, lab, stats, _ = cv2.connectedComponentsWithStats(binm, 8)
        pruned = np.zeros_like(binm)
        for i in range(1, num):
            if stats[i, cv2.CC_STAT_AREA] >= 20:
                pruned[lab==i] = 255
        binm = pruned

    return (binm//255).astype(np.uint8)   # 0/1

# ---- 4) 視覺化（固定大小，避免疊圖維度不合）----
def make_panel(img, prob, pred01, gt01=None, target_hw=None):
    H,W = img.shape[:2]
    if target_hw is None:
        target_hw = (H,W)

    img_r  = resize_like(img,  target_hw, is_mask=False)
    heat_r = resize_like(colorize(prob), target_hw, is_mask=False)
    pred_r = resize_like((pred01*255).astype(np.uint8), target_hw, is_mask=True)
    pred_r3 = cv2.cvtColor(pred_r, cv2.COLOR_GRAY2RGB)

    row1 = np.hstack([img_r, heat_r])
    row2 = np.hstack([pred_r3, np.zeros_like(img_r)])

    if gt01 is not None:
        gt_r  = resize_like((gt01*255).astype(np.uint8), target_hw, is_mask=True)
        gt_r3 = cv2.cvtColor(gt_r, cv2.COLOR_GRAY2RGB)
        row2 = np.hstack([pred_r3, gt_r3])

    panel = np.vstack([row1, row2])
    return panel

# ---- 5) 目錄推論（含 tqdm）----
def infer_dir(img_dir, out_dir, tile=256, overlap=0.5, thr=0.85,
              gt_dir=None, do_post=True, remove_small=80, do_skeleton=True,
              use_tta=True):
    img_dir = Path(img_dir); out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    img_paths = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".png",".jpg",".jpeg",".bmp",".webp"}])
    assert len(img_paths)>0, f"找不到影像在 {img_dir}"

    for p in tqdm(img_paths, desc="Inference"):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        prob = slide_predict(model, img, tile=tile, overlap=overlap, use_tta=use_tta)

        if do_post:
            pred01 = post_process(prob, thr=thr, remove_small=remove_small, do_skeleton=do_skeleton)
        else:
            pred01 = (prob>=thr).astype(np.uint8)

        gt01 = None
        if gt_dir is not None:
            gpath = Path(gt_dir)/ (p.stem + ".png")
            if gpath.exists():
                g = cv2.imread(str(gpath), cv2.IMREAD_GRAYSCALE)
                gt01 = (g>127).astype(np.uint8)

        # 固定把面板輸出為與原圖同大小的四宮格
        panel = make_panel(img, prob, pred01, gt01=gt01, target_hw=img.shape[:2])
        cv2.imwrite(str(out_dir/f"{p.stem}_viz.png"), cv2.cvtColor(panel, cv2.COLOR_RGB2BGR))

        # 也把二值 mask 存一份
        cv2.imwrite(str(out_dir/f"{p.stem}_pred.png"), (pred01*255).astype(np.uint8))

# ======= 實際執行 =======
IMG_DIR = "/content/TUT/val_img"        # 你的「真實世界」影像資料夾
GT_DIR  = "/content/TUT/val/masks"      # 若沒有 GT 可以設為 None
OUT_DIR = "/content/work_scsegamba/real_viz"

thr = 0.85    # 你驗證集找出的 F1 最佳閾值（可微調 0.80~0.90）
print("使用閾值:", thr)

infer_dir(IMG_DIR, OUT_DIR,
          tile=256, overlap=0.5, thr=thr,
          gt_dir=GT_DIR, do_post=True,
          remove_small=80, do_skeleton=True,
          use_tta=True)

print("完成！輸出在：", OUT_DIR)


使用閾值: 0.85


Inference: 100%|██████████| 139/139 [05:04<00:00,  2.19s/it]

完成！輸出在： /content/work_scsegamba/real_viz


In [ ]:
# ============================================
# SCSegamba 推論（Hann-Weighted Sliding + 形狀後處理）
# 可視化成 2x2: [原圖 | 機率熱圖 ; 預測 | 疊圖]
# ============================================

import os, math, glob, cv2, torch, numpy as np
from pathlib import Path
from tqdm import tqdm

# -------------------
# 0) 載入模型與權重
# -------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

# 你的模型類別需已在記憶體中（來自前面訓練檔）
# from your_code import SCSegamba  # 若在其他檔案，記得匯入
model = SCSegamba(base=16).to(device).eval()

CKPT = "/content/work_scsegamba/scsegamba_best.pth"
state = torch.load(CKPT, map_location=device)
model.load_state_dict(state, strict=True)
print("[ok] checkpoint loaded.")

# -------------------
# 1) 小工具
# -------------------
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def to_chw(img_rgb):
    return torch.from_numpy(img_rgb.transpose(2,0,1)).float()/255.0

def colorize(prob):
    pm = (np.clip(prob,0,1)*255).astype(np.uint8)
    return cv2.applyColorMap(pm, cv2.COLORMAP_JET)[:,:,::-1]  # BGR->RGB

def overlay_mask(img_rgb, pred01, alpha=0.45):
    out = img_rgb.copy()
    color = np.zeros_like(img_rgb); color[pred01==1] = [255, 0, 0]  # 紅
    out = (img_rgb*(1-alpha) + color*alpha).astype(np.uint8)
    return out

def hann2d(h, w):
    # 2D Hann window，用於 tile 權重加權
    wy = np.hanning(h)
    wx = np.hanning(w)
    w2 = np.outer(wy, wx).astype(np.float32)
    w2 = np.clip(w2, 1e-4, 1.0)    # 避免 0 導致除法不穩
    return w2

@torch.no_grad()
def _tta_logits(m, x):
    # 4-flip TTA
    y  = m(x)
    y += torch.flip(m(torch.flip(x, [-1])), [-1])
    y += torch.flip(m(torch.flip(x, [-2])), [-2])
    y += torch.flip(m(torch.flip(x, [-2,-1])), [-2,-1])
    return y / 4.0

@torch.no_grad()
def slide_predict_weighted(m, img_chw, tile=256, overlap=0.5, use_tta=True):
    m.eval()
    x = img_chw.to(device).unsqueeze(0)   # [1,3,H,W]
    _,_,H,W = x.shape
    if H<=tile and W<=tile:
        logits = _tta_logits(m, x) if use_tta else m(x)
        return torch.sigmoid(logits)[0,0].cpu().numpy()

    stride = int(tile*(1-overlap))
    pad_h = (int(np.ceil((H-tile)/stride))*stride + tile - H) if H>tile else (tile-H)
    pad_w = (int(np.ceil((W-tile)/stride))*stride + tile - W) if W>tile else (tile-W)

    xpad = torch.nn.functional.pad(x, (0,pad_w,0,pad_h), mode='reflect')
    Hp, Wp = xpad.shape[-2:]
    weight = hann2d(tile, tile)  # [tile, tile]
    w_t = torch.from_numpy(weight)[None,None].to(device)  # [1,1,tile,tile]

    out = torch.zeros((1,1,Hp,Wp), device=device)
    wsum= torch.zeros((1,1,Hp,Wp), device=device)

    for top in range(0, Hp - tile + 1, stride):
        for left in range(0, Wp - tile + 1, stride):
            crop = xpad[:, :, top:top+tile, left:left+tile]
            logits = _tta_logits(m, crop) if use_tta else m(crop)
            prob   = torch.sigmoid(logits)
            out[:,:,top:top+tile, left:left+tile]  += prob * w_t
            wsum[:,:,top:top+tile, left:left+tile] += w_t

    out = out / torch.clamp(wsum, min=1e-6)
    out = out[:,:,:H,:W]
    return out[0,0].detach().float().cpu().numpy()

# -------------------
# 2) 形狀後處理（去 blob、保留細長）
# -------------------
def clean_pred(prob, thr=0.85, min_area=40, min_ar=3.0, open_iter=1, close_iter=1):
    """prob -> binary，去雜點、面積過小 & 非細長者"""
    binm = (prob >= thr).astype(np.uint8)
    k3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
    if open_iter>0:
        binm = cv2.morphologyEx(binm, cv2.MORPH_OPEN, k3, iterations=open_iter)
    if close_iter>0:
        binm = cv2.morphologyEx(binm, cv2.MORPH_CLOSE, k3, iterations=close_iter)

    n, lab, stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
    out = np.zeros_like(binm)
    for i in range(1, n):
        area = stats[i, cv2.CC_STAT_AREA]
        if area < min_area:
            continue
        ys, xs = np.where(lab==i)
        if len(xs) < 5:    # minAreaRect 至少要一些點
            continue
        pts = np.stack([xs, ys], axis=1).astype(np.float32)
        rect = cv2.minAreaRect(pts)
        (w,h) = rect[1]
        long_side = max(w,h); short_side = max(1.0, min(w,h))
        ar = long_side / short_side
        if ar < min_ar:
            continue
        out[lab==i] = 1
    return out

# -------------------
# 3) 圖片可視化：2x2 畫布
# -------------------
def save_quad(img_rgb, prob, pred01, out_path):
    H,W = img_rgb.shape[:2]
    heat = colorize(prob)
    over = overlay_mask(img_rgb, pred01, alpha=0.45)
    pred_rgb = (pred01*255).astype(np.uint8)[...,None].repeat(3,2)

    # 用畫布避免尺寸不一致
    canvas = np.zeros((2*H, 2*W, 3), dtype=np.uint8)
    canvas[0:H, 0:W]     = img_rgb
    canvas[0:H, W:2*W]   = cv2.resize(heat, (W, H), interpolation=cv2.INTER_NEAREST)
    canvas[H:2*H, 0:W]   = pred_rgb
    canvas[H:2*H, W:2*W] = over

    cv2.imwrite(out_path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))

# -------------------
# 4) 目錄推論（含進度條）
# -------------------
def infer_dir(
    img_dir, out_dir, tile=256, overlap=0.5, thr=0.85,
    do_post=True, min_area=40, min_ar=3.0, open_iter=1, close_iter=1
):
    img_dir = Path(img_dir)
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    paths = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
    assert len(paths)>0, f"找不到影像：{img_dir}"

    for p in tqdm(paths, desc="Infer"):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        prob = slide_predict_weighted(model, to_chw(img), tile=tile, overlap=overlap, use_tta=True)
        if do_post:
            pred = clean_pred(prob, thr=thr, min_area=min_area, min_ar=min_ar,
                              open_iter=open_iter, close_iter=close_iter)
        else:
            pred = (prob >= thr).astype(np.uint8)

        save_quad(img, prob, pred, str(out_dir/f"{p.stem}_viz.png"))

    print("Done. 圖檔在：", str(out_dir))

# -------------------
# 5) 實際執行
# -------------------
IMG_DIR = "/content/TUT/val_img"                   # 你的實際照片資料夾
OUT_DIR = "/content/work_scsegamba/real_viz_fixed" # 輸出資料夾

# 若想更保守（更少偽陽性）→ 調高 thr、min_ar、min_area
# 若想更完整（更少漏檢）→ 調低 thr，或把 close_iter 調大
infer_dir(
    IMG_DIR, OUT_DIR,
    tile=256, overlap=0.5,
    thr=0.85,
    do_post=True,
    min_area=50,    # 依資料調整
    min_ar=4.0,     # 形狀先驗：細長（越大越嚴格）
    open_iter=1, close_iter=1
)


[ok] checkpoint loaded.


Infer: 100%|██████████| 139/139 [05:24<00:00,  2.33s/it]

Done. 圖檔在： /content/work_scsegamba/real_viz_fixed


# 6652

In [ ]:
# =================== SCSegamba — paper-aligned + (ours) Focal-Tversky & Edge-Aux ===================
# 變更要點：
#   (1) 新增 Focal-Tversky 與 class-balanced BCE, Dice 的組合損失（針對極端失衡）
#   (2) 新增邊界輔助頭（Edge-Aux），邊界標籤以形態學梯度由 GT 即時生成
#   (3) 輸出頭 bias 以資料先驗（mask/edge）分別設定；DySample offset 加 L2 正則
# 其他沿用你原本版本：GBC、SS2D(Mamba/GRU fallback)、PAF、MFS(DySample)、NaN-proof、強制 FP32 in SS2D
# ==================================================================================================

# ---- 0) 依賴 & Mamba 後備 ----
# !pip -q install albumentations==1.4.7 opencv-python==4.10.0.84

import importlib, os, time, random, math
from pathlib import Path
import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

SEED=2025
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

MAMBA_AVAILABLE=True
try:
    importlib.import_module("mamba_ssm")
except:
    MAMBA_AVAILABLE=False
    # 允許安裝失敗時退回 GRU，不中斷
    try:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "mamba-ssm[causal-conv1d]>=2.2.4"], check=False)
        importlib.import_module("mamba_ssm"); MAMBA_AVAILABLE=True
    except:
        MAMBA_AVAILABLE=False

if MAMBA_AVAILABLE:
    from mamba_ssm import Mamba
else:
    print("[warn] mamba-ssm 不可用，改用 GRU 後備（可跑、較慢）")
    class Mamba(nn.Module):
        def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
            super().__init__(); self.rnn=nn.GRU(d_model,d_model,batch_first=True)
        def forward(self,x): y,_=self.rnn(x); return y

# ---- 1) 資料偵測/準備（與你原版本一致，略微整理）----
IMG_EXT={".png",".jpg",".jpeg",".bmp",".webp"}
IMAGE_DIR_CAND=["images","image","img","imgs","JPEGImages"]
MASK_DIR_CAND=["masks","mask","labels","label","ann","anno","annotations","gt","groundtruth","ground-truth"]
SPLIT_CAND=["train","val","valid","validation","test"]

def _find_first_subdir(base: Path, cands):
    for n in cands:
        p=base/n
        if p.exists() and any(p.glob("*")): return p
    return None

def detect_or_prepare_dataset(root="/content/TUT"):
    root=Path(root); root.mkdir(parents=True, exist_ok=True)
    found={}
    for sp in SPLIT_CAND:
        d=root/sp
        if d.exists():
            img=_find_first_subdir(d, IMAGE_DIR_CAND)
            msk=_find_first_subdir(d, MASK_DIR_CAND)
            if img and msk: found[sp]=(img,msk)
    if ("train" in found) and any(k in found for k in ["val","valid","validation"]):
        if "val" not in found:
            found["val"]=found.get("validation", found.get("valid"))
        print(f"[info] 偵測到現有 train/val 結構：{root}")
        return str(root)

    img_dir=_find_first_subdir(root, IMAGE_DIR_CAND)
    msk_dir=_find_first_subdir(root, MASK_DIR_CAND)
    if img_dir and msk_dir:
        print(f"[info] 偵測到根目錄 images+masks，建立 train/val：{root}")
        imgs=sorted([p for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT]); assert len(imgs)>0
        for sp in ["train","val"]:
            (root/sp/"images").mkdir(parents=True, exist_ok=True)
            (root/sp/"masks").mkdir(parents=True, exist_ok=True)
        n=len(imgs); nv=max(1,int(n*0.1)); valset=set(imgs[-nv:])
        for p in imgs:
            q=msk_dir/p.name
            if not q.exists():
                q_alt=None
                for ext in IMG_EXT:
                    cand=msk_dir/(p.stem+ext)
                    if cand.exists(): q_alt=cand; break
                if q_alt is None: continue
                q=q_alt
            dst="val" if p in valset else "train"
            di=root/dst/"images"/p.name; dm=root/dst/"masks"/q.name
            if not di.exists(): os.link(p, di)
            if not dm.exists(): os.link(q, dm)
        return str(root)

    print("[info] 未偵測到資料；合成迷你資料（僅 smoke test）…")
    def synth(H=512,W=512):
        img=(np.random.rand(H,W,3)*40+100).astype(np.uint8); img=cv2.GaussianBlur(img,(0,0),1.5)
        m=np.zeros((H,W),np.uint8)
        for _ in range(np.random.randint(1,3)):
            x1,y1=np.random.randint(0,W),np.random.randint(0,H)
            x2,y2=np.random.randint(0,W),np.random.randint(0,H)
            cv2.line(m,(x1,y1),(x2,y2),255,thickness=np.random.randint(1,3))
        m=cv2.GaussianBlur(m,(3,3),0); m=(m>127).astype(np.uint8)*255
        return img,m
    for sp,n in [("train",24),("val",6)]:
        (root/sp/"images").mkdir(parents=True, exist_ok=True)
        (root/sp/"masks").mkdir(parents=True, exist_ok=True)
        for i in range(n):
            x,y=synth(512,512)
            cv2.imwrite(str(root/sp/"images"/f"{i:05d}.png"), cv2.cvtColor(x,cv2.COLOR_RGB2BGR))
            cv2.imwrite(str(root/sp/"masks"/f"{i:05d}.png"), y)
    print("[info] 合成完成：", root)
    return str(root)

# ---- 2) Dataset（保留你的正中心裁切；val 單工裁切到 crop 或填邊）----
def _mask_to01(a, invert=False):
    a=(a>127).astype(np.float32)
    return 1.0-a if invert else a

def _ensure_multiple_of_8(h,w):
    H=(h//8)*8; W=(w//8)*8
    if H==0: H=8
    if W==0: W=8
    return H,W

def _center_crop_to(h,w,ch,cw,cx,cy):
    top=max(0, min(h-ch, cy-ch//2)); left=max(0, min(w-cw, cx-cw//2))
    return int(top), int(left)

class CrackSegSet(Dataset):
    def __init__(self, root, split="train", crop=256, auto_invert=True, pos_center=True, tries=30):
        root=Path(root)/split
        img_dir=_find_first_subdir(root, IMAGE_DIR_CAND)
        msk_dir=_find_first_subdir(root, MASK_DIR_CAND)
        assert img_dir and msk_dir, f"{root} 缺 images 或 masks/labels"
        self.X=sorted([str(p) for p in img_dir.glob("*") if p.suffix.lower() in IMG_EXT])
        self.Y=[]
        for p in self.X:
            q=msk_dir/Path(p).name
            if not q.exists():
                for ext in IMG_EXT:
                    cand=msk_dir/(Path(p).stem+ext)
                    if cand.exists(): q=cand; break
            if q.exists(): self.Y.append(str(q))
        self.split=split; self.crop=crop; self.pos_center=(pos_center and split=="train"); self.tries=tries

        self.tf=A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Affine(scale=(0.95,1.05), rotate=(-8,8), translate_percent=(0.0,0.03), p=0.5),
            A.RandomBrightnessContrast(0.08,0.08,p=0.3),
        ])

        self._invert=False; pos=0.0; cnt=0
        self._has_pos_idx=[]
        for i in range(len(self.X)):
            y=np.array(Image.open(self.Y[i]).convert("L"))
            m=(y>127).astype(np.uint8); r=m.mean()
            if r>0: self._has_pos_idx.append(i)
            if cnt<32: pos+=r; cnt+=1
        est= pos/cnt if cnt>0 else 0.0
        if auto_invert and est>0.5: self._invert=True
        print(f"[{split}] samples={len(self.X)}  est_pos_ratio={est:.4f}  invert={self._invert}")

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        if self.split=="train" and self.pos_center and self._has_pos_idx:
            if random.random()<0.8:
                i=random.choice(self._has_pos_idx)

        x=np.array(Image.open(self.X[i]).convert("RGB"))
        y=np.array(Image.open(self.Y[i]).convert("L"))
        y01=_mask_to01(y, invert=self._invert)

        H,W=y01.shape

        if self.split=="train":
            ch,cw=self.crop,self.crop
            if H<ch or W<cw:
                ph=max(0,ch-H); pw=max(0,cw-W)
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y01=cv2.copyMakeBorder(y01,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                H,W=y01.shape
            if self.pos_center and (y01.sum()>0):
                ys,xs=np.where(y01>0.5)
                cy=int(np.median(ys)); cx=int(np.median(xs))
                top,left=_center_crop_to(H,W,ch,cw,cx,cy)
            else:
                top=random.randint(0,H-ch); left=random.randint(0,W-cw)
            xc=x[top:top+ch, left:left+cw]; yc=y01[top:top+ch, left:left+cw]
            aug=self.tf(image=xc, mask=(yc*255).astype(np.uint8))
            x=aug["image"]; y01=(aug["mask"]>127).astype(np.float32)

        else:
            H2,W2=_ensure_multiple_of_8(H,W)
            if H2!=H or W2!=W:
                top=(H-H2)//2; left=(W-W2)//2
                x=x[top:top+H2, left:left+W2]
                y01=y01[top:top+H2, left:left+W2]
                H,W=H2,W2
            ch,cw=self.crop,self.crop
            if H>=ch and W>=cw:
                if y01.sum()>0:
                    ys,xs=np.where(y01>0.5)
                    cy=int(np.median(ys)); cx=int(np.median(xs))
                    top,left=_center_crop_to(H,W,ch,cw,cx,cy)
                else:
                    top=(H-ch)//2; left=(W-cw)//2
                x=x[top:top+ch, left:left+cw]; y01=y01[top:top+ch, left:left+cw]
            else:
                ph=max(0,ch-H); pw=max(0,cw-W)
                x=cv2.copyMakeBorder(x,0,ph,0,pw,cv2.BORDER_REFLECT_101)
                y01=cv2.copyMakeBorder(y01,0,ph,0,pw,cv2.BORDER_REFLECT_101)

        x=torch.from_numpy(x.transpose(2,0,1)).float()/255.0
        y=torch.from_numpy(y01[None]).float()
        return x,y

# ---- 3) 模型主體（與你對齊版本一致；MFS head 內新增 edge 分支）----
def conv_bn_act(cin, cout, k=3, s=1, d=1, groups=1, act=nn.SiLU):
    pad=((k-1)//2)*d
    return nn.Sequential(nn.Conv2d(cin,cout,k,s,pad,dilation=d,groups=groups,bias=False),
                         nn.BatchNorm2d(cout), act(inplace=True))

class PosEnc2D(nn.Module):
    def __init__(self,C,temperature=10000.0):
        super().__init__(); self.C=C; self.temperature=temperature
    def forward(self,x):
        B,C,H,W=x.shape
        y,xg=torch.meshgrid(torch.arange(H,device=x.device), torch.arange(W,device=x.device), indexing='ij')
        n=max(C//4,1); denom=max(C//2,1)
        omega=torch.arange(n,device=x.device).float()
        omega=1.0/(self.temperature**(2*omega/denom))
        y=y[...,None]*omega; xg=xg[...,None]*omega
        pe=torch.cat([torch.sin(xg),torch.cos(xg),torch.sin(y),torch.cos(y)],-1)
        pe=pe.permute(2,0,1).unsqueeze(0)
        if pe.shape[1]!=C:
            if pe.shape[1]<C: pe=torch.cat([pe, torch.zeros((1,C-pe.shape[1],H,W),device=x.device)],1)
            else: pe=pe[:,:C]
        return x+pe

class BottConv(nn.Module):
    def __init__(self,c,r=4,d=1):
        super().__init__(); mid=max(c//r,8)
        self.reduce=conv_bn_act(c,mid,1,1,1)
        self.dw    =conv_bn_act(mid,mid,3,1,d,groups=mid)
        self.expand=conv_bn_act(mid,c,1,1,1)
    def forward(self,x): return self.expand(self.dw(self.reduce(x)))

class GBC(nn.Module):
    def __init__(self,c,dil=2):
        super().__init__()
        self.f1=nn.Conv2d(c,c,1); self.bn1=nn.BatchNorm2d(c)
        self.bc1=BottConv(c,4,dil); self.bn2=nn.BatchNorm2d(c)
        self.bc2=BottConv(c,4,dil); self.bc3=BottConv(c,4,1); self.bn3=nn.BatchNorm2d(c)
        self.act=nn.ReLU(inplace=True)
    def forward(self,x):
        g1=self.act(self.bn1(self.f1(x)))
        x1=self.act(self.bn2(self.bc1(g1)))
        g2=self.act(self.bn2(self.bc2(x1)))
        m=g1*g2
        y=self.act(self.bn3(self.bc3(m)))
        return y+x

def _snake_pairs(H,W,rev=False):
    rows=list(range(H))[::-1] if rev else list(range(H))
    idx=[]
    for r in rows:
        if r%2==0: idx.extend([(r,c) for c in range(W)])
        else: idx.extend([(r,c) for c in range(W-1,-1,-1)])
    return idx

def _diag_pairs(H,W,anti=False):
    idx=[]
    if not anti:
        for s in range(H+W-1):
            d=[]
            for i in range(max(0,s-(W-1)), min(H-1,s)+1):
                j=s-i; d.append((i,j))
            if s%2==0: d=d[::-1]
            idx.extend(d)
    else:
        for s in range(H+W-1):
            d=[]
            for i in range(max(0,s-(W-1)), min(H-1,s)+1):
                j=(W-1)-(s-i); d.append((i,j))
            if s%2==0: d=d[::-1]
            idx.extend(d)
    return idx

class SS2D_Mamba(nn.Module):
    def __init__(self,c,d_state=16,d_conv=4,expand=2):
        super().__init__()
        self.norm=nn.BatchNorm2d(c)
        self.m1=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m2=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m3=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self.m4=Mamba(d_model=c,d_state=d_state,d_conv=d_conv,expand=expand)
        self._cache={}
    def _rrcc(self,H,W,route,device):
        k=(H,W,route)
        if k not in self._cache:
            if route==0: pairs=_snake_pairs(H,W,False)
            elif route==1: pairs=_snake_pairs(H,W,True)
            elif route==2: pairs=_diag_pairs(H,W,False)
            else: pairs=_diag_pairs(H,W,True)
            arr=np.array(pairs, dtype=np.int64)
            self._cache[k]=(torch.from_numpy(arr[:,0]), torch.from_numpy(arr[:,1]))
        rr,cc=self._cache[k]
        return rr.to(device), cc.to(device)
    def forward(self,x):
        # 強制 FP32，避免 AMP 在序列上溢位
        with torch.amp.autocast('cuda', enabled=False):
            x32 = x.to(torch.float32)
            x32 = self.norm(x32)
            B,C,H,W=x32.shape; outs=[]
            for k,m in enumerate([self.m1,self.m2,self.m3,self.m4]):
                rr,cc=self._rrcc(H,W,k,x32.device)
                seq=x32[:,:,rr,cc].permute(0,2,1).contiguous()   # [B, L, C]
                yseq=m(seq)                                      # [B, L, C]
                y=torch.zeros((B,C,H,W), device=x32.device, dtype=yseq.dtype)
                y[:,:,rr,cc]=yseq.permute(0,2,1).contiguous()
                outs.append(y)
            y_sum = sum(outs)
        return y_sum.to(dtype=x.dtype)

class PAFusion(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.gate=nn.Sequential(nn.Conv2d(c*2,c,1), nn.Sigmoid())
        self.lin1=nn.Conv2d(c,c,1); self.gn=nn.GroupNorm(min(32,c),c); self.lin2=nn.Conv2d(c,c,1)
    def forward(self,x,y):
        a=self.gate(torch.cat([x,y],1))
        z=a*y+(1-a)*x
        z=self.lin2(self.gn(self.lin1(z)))
        return z+x

class SAVSS(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.pre_gbc=GBC(c,2)
        self.ss2d=SS2D_Mamba(c)
        self.paf=PAFusion(c)
    def forward(self,x):
        x1=self.pre_gbc(x)
        y=self.ss2d(x1)
        z=self.paf(x1,y)
        return z

class DySample2x(nn.Module):
    def __init__(self,c,max_off=1.0):
        super().__init__()
        self.offset=nn.Conv2d(c,2,3,1,1)
        nn.init.zeros_(self.offset.weight); nn.init.zeros_(self.offset.bias)
        self.max_off=max_off; self._grid={}
    def _base(self,H,W,device):
        k=(H,W,str(device))
        if k not in self._grid:
            yy,xx=torch.meshgrid(torch.linspace(-1,1,2*H,device=device),
                                 torch.linspace(-1,1,2*W,device=device), indexing='ij')
            self._grid[k]=torch.stack([xx,yy],-1).unsqueeze(0)   # [1,2H,2W,2]
        return self._grid[k]
    def forward(self,x):
        B,C,H,W=x.shape
        base=self._base(H,W,x.device).repeat(B,1,1,1)            # [B,2H,2W,2]
        off=torch.tanh(self.offset(x))                           # [B,2,H,W]
        off=F.interpolate(off,size=(2*H,2*W),mode='bilinear',align_corners=False)
        dx=(off[:,0]/W)*self.max_off; dy=(off[:,1]/H)*self.max_off
        grid=torch.stack([base[...,0]+dx, base[...,1]+dy],-1).clamp(-1,1)
        return F.grid_sample(x, grid, mode='bilinear', padding_mode='border', align_corners=False)

class MLP_Head(nn.Module):
    def __init__(self,c):
        super().__init__()
        self.mlp=nn.Sequential(nn.Conv2d(c,c,1), nn.SiLU(True), nn.Conv2d(c,c,1), nn.SiLU(True))
    def forward(self,x): return self.mlp(x)

class MFS_Head(nn.Module):
    # ★ 這裡加入 Edge 分支（edge_head）
    def __init__(self,c1,c2,c3,c4,out_c=1):
        super().__init__()
        self.mlp1,self.mlp2,self.mlp3,self.mlp4 = MLP_Head(c1),MLP_Head(c2),MLP_Head(c3),MLP_Head(c4)
        self.up2=DySample2x(c2); self.up3a=DySample2x(c3); self.up3b=DySample2x(c3)
        self.up4a=DySample2x(c4); self.up4b=DySample2x(c4); self.up4c=DySample2x(c4)
        self.a2=nn.Conv2d(c2,c1,1); self.a3=nn.Conv2d(c3,c1,1); self.a4=nn.Conv2d(c4,c1,1)
        self.gbc=GBC(4*c1,1); self.conv=conv_bn_act(4*c1,2*c1,3)
        self.mlp_o=nn.Sequential(nn.Conv2d(2*c1,2*c1,1), nn.SiLU(True), nn.Conv2d(2*c1,c1,1), nn.SiLU(True))
        self.head     = nn.Conv2d(c1,out_c,1)   # 主遮罩
        self.edge_head= nn.Conv2d(c1,1,1)       # ★ 邊界分支
    def forward(self,F1,F2,F3,F4):
        F1u=self.mlp1(F1)
        F2u=self.a2(self.up2(self.mlp2(F2)))
        F3u=self.a3(self.up3b(self.up3a(self.mlp3(F3))))
        F4u=self.a4(self.up4c(self.up4b(self.up4a(self.mlp4(F4)))))
        cat=torch.cat([F1u,F2u,F3u,F4u],1)
        o1=self.conv(self.gbc(cat))
        o =self.mlp_o(o1)
        return self.head(o), self.edge_head(o)  # 回傳 (mask_logits, edge_logits)

class SCSegamba(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        c1,c2,c3,c4=base,base*2,base*4,base*8
        self.embed=conv_bn_act(3,c1,3); self.pe=PosEnc2D(c1); self.b1=SAVSS(c1)
        self.e2=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c1,c2,3)); self.b2=SAVSS(c2)
        self.e3=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c2,c3,3)); self.b3=SAVSS(c3)
        self.e4=nn.Sequential(nn.MaxPool2d(2), conv_bn_act(c3,c4,3)); self.b4=SAVSS(c4)
        self.head=MFS_Head(c1,c2,c3,c4,1)
        self._n_params=sum(p.numel() for p in self.parameters() if p.requires_grad)
    def forward(self,x):
        f1=self.b1(self.pe(self.embed(x)))
        f2=self.b2(self.e2(f1))
        f3=self.b3(self.e3(f2))
        f4=self.b4(self.e4(f3))
        return self.head(f1,f2,f3,f4)  # (mask_logits, edge_logits)

# ---- 4) Loss / Metrics / 可視化 ----
class ComboSegLoss(nn.Module):
    """L = λ_bce·BCE(pos-weighted) + λ_dice·Dice + λ_ft·Focal-Tversky"""
    def __init__(self, lambda_bce=1.0, lambda_dice=1.0, lambda_ft=0.5,
                 pos_weight=None, tv_alpha=0.3, tv_beta=0.7, tv_gamma=4.0, eps=1e-7):
        super().__init__()
        self.lb=lambda_bce; self.ld=lambda_dice; self.lf=lambda_ft
        self.eps=eps
        self.bce=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.ta=tv_alpha; self.tb=tv_beta; self.tg=tv_gamma
    def forward(self, logits, y):
        # BCE
        bce=self.bce(logits,y)
        # Dice
        p=torch.sigmoid(logits)
        inter=(p*y).sum((1,2,3)); summ=(p+y).sum((1,2,3))
        dice_loss=1-(2*inter+self.eps)/(summ+self.eps)
        dice_loss=torch.nan_to_num(dice_loss, nan=0.0, posinf=1.0, neginf=0.0).mean()
        # Focal-Tversky
        tp=(p*y).sum((1,2,3))
        fp=(p*(1-y)).sum((1,2,3))
        fn=((1-p)*y).sum((1,2,3))
        tv = tp/(tp + self.ta*fp + self.tb*fn + self.eps)
        ft = torch.pow(1.0 - tv, self.tg).mean()
        # 總和
        loss = self.lb*bce + self.ld*dice_loss + self.lf*ft
        return loss, bce.detach(), dice_loss.detach(), ft.detach()

@torch.no_grad()
def metrics_at(logits,y,thr=0.5,eps=1e-7):
    p=(torch.sigmoid(logits)>thr).float()
    tp=(p*y).sum().item(); fp=(p*(1-y)).sum().item(); fn=((1-p)*y).sum().item()
    iou=tp/(tp+fp+fn+eps); f1=2*tp/(2*tp+fp+fn+eps)
    return iou,f1

@torch.no_grad()
def as_rgb(a: np.ndarray, to_size=None) -> np.ndarray:
    a = np.nan_to_num(a, nan=0.0, posinf=255.0, neginf=0.0).astype(np.uint8)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=-1)
    elif a.ndim == 3 and a.shape[2] == 1:
        a = np.repeat(a, 3, axis=-1)
    elif a.ndim == 3 and a.shape[2] >= 3:
        a = a[..., :3]
    else:
        a = np.repeat(a[..., None], 3, axis=-1)
    if to_size is not None:
        a = cv2.resize(a, (to_size[1], to_size[0]), interpolation=cv2.INTER_NEAREST)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=-1)
    return a

@torch.no_grad()
def save_viz(model, ds, out_dir, k=6):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    device=next(model.parameters()).device
    model.eval()
    n=min(k,len(ds))
    for i in range(n):
        x,y=ds[i]; x=x.unsqueeze(0).to(device); y=y.to(device)
        mask_logits, edge_logits = model(x)
        if not torch.isfinite(mask_logits).all().item(): print("[viz] 非有限 logits，略過"); continue
        prob=torch.sigmoid(mask_logits); edge_prob=torch.sigmoid(edge_logits)

        img=(x[0].permute(1,2,0).cpu().numpy()*255.0)
        prb=(prob[0,0].clamp(0,1).cpu().numpy()*255.0)
        msk=(y[0,0].clamp(0,1).cpu().numpy()*255.0)
        epr=(edge_prob[0,0].clamp(0,1).cpu().numpy()*255.0)

        H,W=img.shape[:2]
        vis=np.vstack([
            as_rgb(img,(H,W)),
            as_rgb(prb,(H,W)),
            as_rgb(msk,(H,W)),
            as_rgb(epr,(H,W)),
        ])
        cv2.imwrite(str(Path(out_dir)/f"viz_{i:02d}.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))

# ---- 5) 先驗估計／邊界 GT 生成 ----
def estimate_pos_ratio_full(ds):
    s=0.0; c=0
    for i in range(len(ds.X)):
        y=np.array(Image.open(ds.Y[i]).convert("L"))
        s+=(y>127).mean(); c+=1
    return (s/c) if c>0 else 0.01

def _edge_from_y_torch(y01):
    # 以 (maxpool - minpool) 近似形態學梯度；y01 ∈ [0,1], shape [B,1,H,W]
    y_max = F.max_pool2d(y01, kernel_size=3, stride=1, padding=1)
    y_min = -F.max_pool2d(-y01, kernel_size=3, stride=1, padding=1)
    e = (y_max - y_min)
    return (e>0).float()

def estimate_edge_ratio_full(ds):
    s=0.0; c=0
    ker=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))
    for i in range(len(ds.X)):
        y=(np.array(Image.open(ds.Y[i]).convert("L"))>127).astype(np.uint8)*255
        dil=cv2.dilate(y,ker,1); ero=cv2.erode(y,ker,1)
        e=((dil-ero)>0).astype(np.uint8)
        s+=e.mean(); c+=1
    return (s/c) if c>0 else 0.005

def _logit(p):
    p=float(np.clip(p, 1e-4, 1-1e-4)); return math.log(p/(1-p))

def set_head_priors(model, p_mask=0.01, p_edge=0.005):
    # 直接設定 MFS_Head 兩個輸出頭的 bias
    if hasattr(model, "head"):
        if hasattr(model.head, "head") and model.head.head.bias is not None:
            model.head.head.bias.data.fill_(_logit(p_mask))
            print(f"[info] mask head bias = logit({p_mask:.4f}) = {_logit(p_mask):.3f}")
        if hasattr(model.head, "edge_head") and model.head.edge_head.bias is not None:
            model.head.edge_head.bias.data.fill_(_logit(p_edge))
            print(f"[info] edge head bias = logit({p_edge:.4f}) = {_logit(p_edge):.3f}")

# ---- 6) 訓練 ----
def train(
    data_root="/content/TUT", work_dir="/content/work_scsegamba",
    base=16, crop=256, batch=6, epochs=30, lr=5e-4, amp=False,
    lambda_edge=0.25, lambda_offset=1e-4
):
    Path(work_dir).mkdir(parents=True, exist_ok=True)
    device="cuda" if torch.cuda.is_available() else "cpu"
    pin = (device=="cuda")
    data_root=detect_or_prepare_dataset(data_root)

    tr=CrackSegSet(data_root,"train",crop,auto_invert=True,pos_center=True,tries=30)
    va=CrackSegSet(data_root,"val",  crop,auto_invert=True,pos_center=False)

    tl=DataLoader(tr,batch,shuffle=True,num_workers=(2 if pin else 0),pin_memory=pin,drop_last=True)
    vl=DataLoader(va,1,shuffle=False,num_workers=0,pin_memory=pin)   # val 單工以穩定

    # 先驗與權重
    p_mask = max(min(estimate_pos_ratio_full(tr),0.49),1e-4)
    p_edge = max(min(estimate_edge_ratio_full(tr),0.49),1e-4)
    pos_w_mask = float(np.clip((1-p_mask)/p_mask, 1.0, 50.0))
    pos_w_edge = float(np.clip((1-p_edge)/p_edge, 1.0, 50.0))
    print(f"[info] 先驗 p_mask≈{p_mask:.4f} → pos_weight={pos_w_mask:.2f}")
    print(f"[info] 先驗 p_edge≈{p_edge:.4f} → pos_weight_edge={pos_w_edge:.2f}")

    model=SCSegamba(base=base).to(device).float()
    set_head_priors(model, p_mask=p_mask, p_edge=p_edge)
    print(f"Model params: {model._n_params/1e6:.2f}M | Mamba: {MAMBA_AVAILABLE}")

    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs,1))

    seg_loss = ComboSegLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_ft=0.5,
                            pos_weight=torch.tensor([pos_w_mask], device=device),
                            tv_alpha=0.3, tv_beta=0.7, tv_gamma=4.0)
    edge_loss= ComboSegLoss(lambda_bce=1.0, lambda_dice=1.0, lambda_ft=0.5,
                            pos_weight=torch.tensor([pos_w_edge], device=device),
                            tv_alpha=0.3, tv_beta=0.7, tv_gamma=4.0)

    use_amp=amp and (device=="cuda")
    scaler=torch.amp.GradScaler('cuda') if use_amp else None

    best_iou=-1; best=str(Path(work_dir)/"scsegamba_best.pth")
    nan_batches=0

    for ep in range(1,epochs+1):
        model.train(); t0=time.time()
        run_main=run_edge=run_bce=run_dice=run_ft=0.0; nstep=0; logit_mean=0.0

        for x,y in tl:
            x,y=x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)

            if use_amp:
                with torch.amp.autocast('cuda', enabled=True):
                    mask_logits, edge_logits = model(x)
                if not torch.isfinite(mask_logits).all().item(): nan_batches+=1; continue
                L_main,bce,dice,ft = seg_loss(mask_logits,y)
                y_edge=_edge_from_y_torch(y)
                L_edge,_,_,_         = edge_loss(edge_logits,y_edge)
                # DySample offset L2
                off_reg = 0.0
                for n,p in model.named_parameters():
                    if "offset" in n: off_reg += (p**2).mean()
                loss = L_main + lambda_edge*L_edge + lambda_offset*off_reg
                if not torch.isfinite(loss).all().item(): nan_batches+=1; continue
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update()
            else:
                mask_logits, edge_logits = model(x)
                if not torch.isfinite(mask_logits).all().item(): nan_batches+=1; continue
                L_main,bce,dice,ft = seg_loss(mask_logits,y)
                y_edge=_edge_from_y_torch(y)
                L_edge,_,_,_         = edge_loss(edge_logits,y_edge)
                off_reg = 0.0
                for n,p in model.named_parameters():
                    if "offset" in n: off_reg += (p**2).mean()
                loss = L_main + lambda_edge*L_edge + lambda_offset*off_reg
                if not torch.isfinite(loss).all().item(): nan_batches+=1; continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            nstep+=1
            run_main+=L_main.detach().item()
            run_edge+=(L_edge.detach().item()*lambda_edge)
            run_bce+=bce.item(); run_dice+=dice.item(); run_ft+=ft.item()
            logit_mean+=mask_logits.mean().detach().item()

        sch.step()

        # ---- 驗證（看主遮罩）----
        model.eval(); iou3=[]; f13=[]; iou5=[]; f15=[]
        with torch.no_grad():
            for x,y in vl:
                x,y=x.to(device), y.to(device)
                logits,_=model(x)
                if not torch.isfinite(logits).all().item(): continue
                a,b=metrics_at(logits,y,thr=0.3); iou3.append(a); f13.append(b)
                a,b=metrics_at(logits,y,thr=0.5); iou5.append(a); f15.append(b)
        miou3=float(np.mean(iou3)) if iou3 else 0.0
        mf13 =float(np.mean(f13)) if f13 else 0.0
        miou5=float(np.mean(iou5)) if iou5 else 0.0
        mf15 =float(np.mean(f15)) if f15 else 0.0

        print(f"Ep {ep:03d}/{epochs} | "
              f"Main={run_main/max(nstep,1):.4f}  Edge(λ)={run_edge/max(nstep,1):.4f} "
              f"[BCE={run_bce/max(nstep,1):.4f}, Dice={run_dice/max(nstep,1):.4f}, FT={run_ft/max(nstep,1):.4f}] "
              f"logit_mean={logit_mean/max(nstep,1):.3f} | "
              f"mIoU@0.3={miou3:.4f} F1@0.3={mf13:.4f} | mIoU@0.5={miou5:.4f} F1@0.5={mf15:.4f} | "
              f"{time.time()-t0:.1f}s  | skipped={nan_batches}")

        if ep%5==0:  # 每 5 個 epoch 存幾張可視化（含 edge prob）
            save_viz(model, va, Path(work_dir)/f"viz_ep{ep}", k=min(6,len(va)))

        if miou5>best_iou:
            best_iou=miou5; torch.save(model.state_dict(), best)

    print(f"[Done] best mIoU@0.5={best_iou:.4f} | saved to {best}")
    return model

# ---- 7) 執行 ----
DATA_ROOT="/content/TUT"
model=train(DATA_ROOT, base=16, crop=256, batch=3, epochs=100, lr=5e-4, amp=False,
            lambda_edge=0.25, lambda_offset=1e-4)
